# Review copy

Historical hand-tracker experiments, original cells 15–28. Extracted from the Drive notebook experiments_and _tests.ipynb without changing code or Markdown. Outputs and execution counts were removed to reduce file size. This part is for review and is not standalone. Original cell order is preserved.

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert 'vram_images' in globals(), "Dataset tensors missing from memory."

# Nominal camera capture rate (Re:InterHand source video is 30 FPS)
CAPTURE_FPS = 30.0

# -------------------------------------------------------------------------
# 1. TEMPORAL METADATA EXTRACTION & SEGMENT ORDERING
# -------------------------------------------------------------------------
def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

frame_ids = np.array([extract_frame_id(r["image_path"]) for r in manifest_rows])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

for seg in seg_to_indices:
    seg_to_indices[seg].sort(key=lambda i: frame_ids[i])

# -------------------------------------------------------------------------
# 2. FROZEN PERCEPTION BACKBONE (LCF ON SCALAR SOBEL)
# -------------------------------------------------------------------------
class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=1)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 3. MANIFOLD-CONSTRAINED DIRECTIONAL FILTER ON S²
# -------------------------------------------------------------------------
def slerp(u0: np.ndarray, u1: np.ndarray, t: float) -> np.ndarray:
    """Geodesic spherical interpolation on S²."""
    dot = np.clip(np.dot(u0, u1), -1.0, 1.0)
    theta = np.arccos(dot)
    if theta < 1e-4:
        blend = (1.0 - t) * u0 + t * u1
        return blend / (np.linalg.norm(blend) + 1e-7)
    sin_theta = np.sin(theta)
    return (np.sin((1.0 - t) * theta) / sin_theta) * u0 + (np.sin(t * theta) / sin_theta) * u1

class AdaptiveDirectionalFilter:
    def __init__(self, fps: float = 30.0):
        self.fps = fps
        # Maximum physiological velocity: 180 deg/s
        self.omega_max = 180.0 * (np.pi / 180.0)

        # Empirical measurement standard deviations per bone in radians
        sigma_degs = np.array([
            22.5, 27.7, 27.3, 33.6,  # Thumb
            14.3, 22.8, 32.8, 34.6,  # Index
            12.7, 22.6, 37.6, 41.1,  # Middle
            13.5, 25.1, 38.9, 43.6,  # Ring
            15.9, 33.0, 44.8, 48.2   # Pinky
        ])
        self.sigma_R = sigma_degs * (np.pi / 180.0)
        self.R_var = self.sigma_R ** 2

        # Continuous process drift rate: rad²/s
        self.q_rate = np.array([
            0.15, 0.20, 0.25, 0.30,  # Thumb
            0.10, 0.20, 0.35, 0.45,  # Index
            0.08, 0.20, 0.35, 0.45,  # Middle
            0.10, 0.20, 0.35, 0.45,  # Ring
            0.12, 0.25, 0.40, 0.50   # Pinky
        ])

    def filter_sequence(self, raw_dirs: np.ndarray, f_ids: np.ndarray):
        T = len(f_ids)
        filtered_dirs = np.zeros_like(raw_dirs)

        u_state = raw_dirs[0].copy()
        P_cov = self.R_var.copy()
        filtered_dirs[0] = u_state

        total_gates = 0
        total_evals = (T - 1) * 20

        for t in range(1, T):
            delta_frame = max(1, f_ids[t] - f_ids[t - 1])
            delta_t = delta_frame / self.fps

            # Continuous process variance update
            P_pred = P_cov + self.q_rate * delta_t
            z_meas = raw_dirs[t]

            u_next = np.zeros((20, 3))
            for b in range(20):
                u_p = u_state[b]
                u_m = z_meas[b]

                # Angular innovation
                dot = np.clip(np.dot(u_p, u_m), -1.0, 1.0)
                theta_innov = np.arccos(dot)

                # Continuous physiological innovation gate
                theta_gate = self.omega_max * delta_t + 3.0 * self.sigma_R[b]

                if theta_innov > theta_gate:
                    total_gates += 1
                    # Clamp measurement along geodesic to gate boundary
                    clamp_ratio = theta_gate / (theta_innov + 1e-7)
                    u_m = slerp(u_p, u_m, clamp_ratio)
                    theta_innov = theta_gate

                # Slerp state update on S²
                K = P_pred[b] / (P_pred[b] + self.R_var[b])
                u_next[b] = slerp(u_p, u_m, K)
                P_cov[b] = (1.0 - K) * P_pred[b]

            u_state = u_next
            filtered_dirs[t] = u_state

        gate_rate = (total_gates / max(1, total_evals)) * 100.0
        return filtered_dirs, gate_rate

# -------------------------------------------------------------------------
# 4. BENCHMARK HARNESS ACROSS 3 HELD-OUT SPLITS
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_raw_mpjpe = []
results_filt_mpjpe = []
results_raw_distal = []
results_filt_distal = []
results_raw_fast = []
results_filt_fast = []
results_gating = []
results_tau_star = []

filter_engine = AdaptiveDirectionalFilter(fps=CAPTURE_FPS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

print("=" * 95)
print(" BENCHMARKING PHYSICAL-TIME DIRECTIONAL FILTER (3 HELD-OUT SPLITS)")
print("=" * 95)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
    model = LatentFactorizationModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)

            x_batch = gpu_extractor(raw_rgb, mode="scalar")
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss, _, _, _ = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    # Chronological validation evaluation per held-out segment
    model.eval()
    val_raw_joints, val_filt_joints, val_gt_joints = [], [], []
    val_displacements = []
    split_gate_rates = []

    with torch.no_grad():
        for seg in val_segs:
            s_indices = seg_to_indices[seg]
            f_ids = frame_ids[s_indices]

            s_raw_dirs, s_gt = [], []
            for i in range(0, len(s_indices), BATCH_SIZE):
                b_idx = s_indices[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                x_batch = gpu_extractor(raw_rgb, mode="scalar")
                _, b_dirs = model(x_batch)
                s_raw_dirs.append(b_dirs.cpu().numpy())
                s_gt.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

            raw_dirs_np = np.concatenate(s_raw_dirs, axis=0)
            gt_joints_np = np.concatenate(s_gt, axis=0)

            filt_dirs_np, gate_pct = filter_engine.filter_sequence(raw_dirs_np, f_ids)
            split_gate_rates.append(gate_pct)

            raw_vecs = torch.tensor(raw_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)
            filt_vecs = torch.tensor(filt_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)

            raw_j = reconstruct_joints_from_bones(raw_vecs).cpu().numpy()
            filt_j = reconstruct_joints_from_bones(filt_vecs).cpu().numpy()

            val_raw_joints.append(raw_j)
            val_filt_joints.append(filt_j)
            val_gt_joints.append(gt_joints_np)

            # Measure GT inter-frame displacement (mm) for fast-motion slicing
            disp_seg = [0.0]
            for t in range(len(s_indices) - 1):
                d = np.linalg.norm(gt_joints_np[t + 1] - gt_joints_np[t], axis=-1).mean()
                disp_seg.append(d)
            val_displacements.append(np.array(disp_seg))

    all_raw = np.concatenate(val_raw_joints, axis=0)
    all_filt = np.concatenate(val_filt_joints, axis=0)
    all_gt = np.concatenate(val_gt_joints, axis=0)
    all_disp = np.concatenate(val_displacements, axis=0)

    # 1. Global MPJPE
    raw_mpjpe = np.linalg.norm(all_raw - all_gt, axis=-1).mean()
    filt_mpjpe = np.linalg.norm(all_filt - all_gt, axis=-1).mean()

    # 2. Distal Direction Error
    b_gt = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]
    u_gt = b_gt / (np.linalg.norm(b_gt, axis=-1, keepdims=True) + 1e-7)

    raw_b = all_raw[:, CHILD_INDICES.cpu().numpy()] - all_raw[:, PARENT_INDICES.cpu().numpy()]
    u_raw = raw_b / (np.linalg.norm(raw_b, axis=-1, keepdims=True) + 1e-7)

    filt_b = all_filt[:, CHILD_INDICES.cpu().numpy()] - all_filt[:, PARENT_INDICES.cpu().numpy()]
    u_filt = filt_b / (np.linalg.norm(filt_b, axis=-1, keepdims=True) + 1e-7)

    deg_raw = np.arccos(np.clip(np.sum(u_raw[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)
    deg_filt = np.arccos(np.clip(np.sum(u_filt[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)

    # 3. Fast-Motion Slice (>95th percentile velocity)
    threshold_95 = np.percentile(all_disp, 95)
    fast_mask = all_disp >= threshold_95
    fast_raw_mpjpe = np.linalg.norm(all_raw[fast_mask] - all_gt[fast_mask], axis=-1).mean()
    fast_filt_mpjpe = np.linalg.norm(all_filt[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    # 4. Empirical Lag Offset Search: tau* = argmin_tau E[P_filt(t), P_gt(t + tau)]
    tau_shifts = [-2, -1, 0, 1, 2]
    tau_errors = []
    for tau in tau_shifts:
        if tau < 0:
            err = np.linalg.norm(all_filt[:tau] - all_gt[-tau:], axis=-1).mean()
        elif tau > 0:
            err = np.linalg.norm(all_filt[tau:] - all_gt[:-tau], axis=-1).mean()
        else:
            err = filt_mpjpe
        tau_errors.append(err)
    best_tau = tau_shifts[int(np.argmin(tau_errors))]

    results_raw_mpjpe.append(raw_mpjpe)
    results_filt_mpjpe.append(filt_mpjpe)
    results_raw_distal.append(deg_raw.mean())
    results_filt_distal.append(deg_filt.mean())
    results_raw_fast.append(fast_raw_mpjpe)
    results_filt_fast.append(fast_filt_mpjpe)
    results_gating.append(np.mean(split_gate_rates))
    results_tau_star.append(best_tau)

    print(f"  [Seed {seed:<3}] MPJPE: {raw_mpjpe:5.2f}->{filt_mpjpe:5.2f} mm | Distal: {deg_raw.mean():4.1f}°->{deg_filt.mean():4.1f}° | Fast(>95%): {fast_raw_mpjpe:5.2f}->{fast_filt_mpjpe:5.2f} mm | Gated: {np.mean(split_gate_rates):4.1f}% | tau*: {best_tau}")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 5. FINAL REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" ADAPTIVE DIRECTIONAL FILTER: MULTI-METRIC PERFORMANCE AUDIT")
print("=" * 105)
print(f"{'Evaluation Metric':<32} | {'Raw Static LCF':<20} | {'Filtered Kinematic':<20} | {'Delta':<12}")
print("-" * 105)

m_raw, s_raw = np.mean(results_raw_mpjpe), np.std(results_raw_mpjpe)
m_flt, s_flt = np.mean(results_filt_mpjpe), np.std(results_filt_mpjpe)
print(f"{'Overall MPJPE (mm)':<32} | {m_raw:5.2f} ± {s_raw:4.2f} mm    | {m_flt:5.2f} ± {s_flt:4.2f} mm    | {m_flt - m_raw:+5.2f} mm")

m_dr, s_dr = np.mean(results_raw_distal), np.std(results_raw_distal)
m_df, s_df = np.mean(results_filt_distal), np.std(results_filt_distal)
print(f"{'Distal Tip Direction Error':<32} | {m_dr:5.2f}° ± {s_dr:4.2f}°     | {m_df:5.2f}° ± {s_df:4.2f}°     | {m_df - m_dr:+5.2f}°")

m_fr, s_fr = np.mean(results_raw_fast), np.std(results_raw_fast)
m_ff, s_ff = np.mean(results_filt_fast), np.std(results_filt_fast)
print(f"{'Fast-Motion MPJPE (>95th %)':<32} | {m_fr:5.2f} ± {s_fr:4.2f} mm    | {m_ff:5.2f} ± {s_ff:4.2f} mm    | {m_ff - m_fr:+5.2f} mm")

print(f"{'Mean Innovation Gating Rate':<32} | {'—':<20} | {np.mean(results_gating):5.2f} %            | —")
print(f"{'Optimal Lag Shift (tau*)':<32} | {'—':<20} | {int(np.median(results_tau_star)):<2} frames            | —")
print("=" * 105)

In [ ]:
# @title
import os
import gc
import csv
import json
import time
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE VERIFICATION & VRAM PREPARATION
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Go to Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Hardware Accelerator : {gpu_name}")
print(f"Available VRAM       : {total_vram_gb:.2f} GB")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure the dataset is extracted.")

# Official Re:InterHand 24 -> Kinematic 21 Joint Index Mapping
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)
OUT_DIM = NUM_JOINTS * 3
TARGET_W, TARGET_H = 256, 192

# -------------------------------------------------------------------------
# 2. GPU SOBEL EXTRACTOR
# -------------------------------------------------------------------------
class GPURepresentationEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx_1 = kx.view(1, 1, 3, 3)
        self.ky_1 = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        # Standard luminance (B, 1, H, W)
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx_1, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky_1, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPURepresentationEngine(DEVICE)

# -------------------------------------------------------------------------
# 3. SELF-HEALING VRAM INGESTION (SAFE UINT8 CACHING)
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("\nPreloading dataset into Tesla T4 VRAM (uint8 compact storage)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_images_list, raw_targets_list, sample_segments_list = [], [], []

    for row in records:
        try:
            img_p = resolve_file(DATA_DIR, row["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, row["orig_kp_right"], "keypoints_right")

            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_images_list.append(rgb_resized.transpose(2, 0, 1))
            raw_targets_list.append(kps_rel)
            sample_segments_list.append(row["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_images_list), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targets_list), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(sample_segments_list)
    unique_segments = np.unique(sample_segments)

    del raw_images_list, raw_targets_list, sample_segments_list
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} existing frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 4. KINEMATIC TOPOLOGY & STRUCTURAL LOSS DEFINITIONS
# -------------------------------------------------------------------------
# 20 directed bone parent-child pairs in 21-joint space
BONE_PAIRS = [
    # Thumb: Wrist -> CMC -> MCP -> IP -> Tip
    (0, 1), (1, 2), (2, 3), (3, 4),
    # Index: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 5), (5, 6), (6, 7), (7, 8),
    # Middle: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 9), (9, 10), (10, 11), (11, 12),
    # Ring: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 13), (13, 14), (14, 15), (15, 16),
    # Pinky: Wrist -> MCP -> PIP -> DIP -> Tip
    (0, 17), (17, 18), (18, 19), (19, 20)
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

class KinematicStructureLoss(nn.Module):
    def __init__(self, lambda_len: float = 0.5, lambda_dir: float = 10.0):
        super().__init__()
        self.lambda_len = lambda_len
        self.lambda_dir = lambda_dir

    def forward(self, pred: torch.Tensor, target: torch.Tensor):
        # Base Coordinate Position Loss (MSE)
        l_pos = F.mse_loss(pred, target)

        # Bone vectors (B, 20, 3)
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        # Bone length consistency (L1 error in mm)
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        # Bone direction angular alignment (Cosine distance)
        eps = 1e-7
        u_pred = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_gt = b_gt / (len_gt.unsqueeze(-1) + eps)
        l_dir = (1.0 - (u_pred * u_gt).sum(dim=-1)).mean()

        total = l_pos + self.lambda_len * l_len + self.lambda_dir * l_dir
        return total

# -------------------------------------------------------------------------
# 5. POSE BACKBONE (1-CHANNEL INPUT)
# -------------------------------------------------------------------------
class PoseBackbone(nn.Module):
    def __init__(self, in_channels: int = 1, out_dim: int = 63):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.head = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(512, out_dim)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.head(torch.flatten(feat, 1))

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS: UNCONSTRAINED MSE VS. KINEMATIC LOSS
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

EXPERIMENTS = [
    ("Unconstrained MSE Loss", False),
    ("Kinematic Structural Loss", True)
]

comparison_matrix = {name: [] for name, _ in EXPERIMENTS}
bone_error_matrix = {name: [] for name, _ in EXPERIMENTS}

print("\n" + "=" * 80)
print(" BENCHMARKING LOSS OBJECTIVES ACROSS 3 SEGMENT SPLITS")
print("=" * 80)

for exp_name, use_kinematic in EXPERIMENTS:
    print(f"\n>>> Running: {exp_name} <<<")
    kin_loss_fn = KinematicStructureLoss()

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        # Strictly identical segment splits across both runs
        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = PoseBackbone(in_channels=1, out_dim=OUT_DIM).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        best_val_mpjpe = float("inf")
        best_val_bone_err = 0.0

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb_batch = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb_batch)

                optimizer.zero_grad()
                pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                loss = kin_loss_fn(pred, y_batch) if use_kinematic else F.mse_loss(pred, y_batch)
                loss.backward()
                optimizer.step()

            # Validation evaluation
            model.eval()
            total_err, total_bone_err, total_pts = 0.0, 0.0, 0
            with torch.no_grad():
                for i in range(0, len(val_indices), BATCH_SIZE):
                    b_idx = val_indices[i:i + BATCH_SIZE]
                    raw_rgb_batch = vram_images[b_idx].float()
                    y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                    x_batch = gpu_extractor(raw_rgb_batch)
                    pred = model(x_batch).view(-1, NUM_JOINTS, 3)

                    # MPJPE (mm)
                    dist = torch.norm(pred - y_batch, dim=-1)
                    total_err += dist.sum().item()
                    total_pts += dist.numel()

                    # Bone length deformation error (mm)
                    b_pred = torch.norm(pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES], dim=-1)
                    b_gt = torch.norm(y_batch[:, CHILD_INDICES] - y_batch[:, PARENT_INDICES], dim=-1)
                    total_bone_err += torch.abs(b_pred - b_gt).sum().item()

            val_mpjpe = total_err / total_pts
            val_bone_err = total_bone_err / (len(val_indices) * len(BONE_PAIRS))

            if val_mpjpe < best_val_mpjpe:
                best_val_mpjpe = val_mpjpe
                best_val_bone_err = val_bone_err

        comparison_matrix[exp_name].append(best_val_mpjpe)
        bone_error_matrix[exp_name].append(best_val_bone_err)
        print(f"  [Seed {seed:<3}] Val MPJPE: {best_val_mpjpe:6.2f} mm | Bone Length Error: {best_val_bone_err:5.2f} mm")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 7. RESULTS SUMMARY
# -------------------------------------------------------------------------
print("\n" + "=" * 80)
print(" LOSS OBJECTIVE COMPARISON: 3-SPLIT SEGMENT-HELD-OUT EVALUATION")
print("=" * 80)
print(f"{'Objective Formulation':<30} | {'Mean MPJPE':<18} | {'Bone Length Error':<18}")
print("-" * 80)
for exp_name, _ in EXPERIMENTS:
    m_pjpe = np.mean(comparison_matrix[exp_name])
    s_pjpe = np.std(comparison_matrix[exp_name])
    m_bone = np.mean(bone_error_matrix[exp_name])
    s_bone = np.std(bone_error_matrix[exp_name])
    print(f"{exp_name:<30} | {m_pjpe:6.2f} ± {s_pjpe:4.2f} mm | {m_bone:5.2f} ± {s_bone:4.2f} mm")
print("=" * 80)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest missing at {MANIFEST_PATH}. Ensure dataset is extracted.")

TARGET_W, TARGET_H = 256, 192
CAPTURE_FPS = 30.0

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Parent-Child Bone Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

# -------------------------------------------------------------------------
# 1. SELF-HEALING VRAM DATA INGESTION
# -------------------------------------------------------------------------
def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str) -> Path:
    p1 = base_dir / rel_path
    if p1.exists():
        return p1
    p2 = base_dir / fallback_subdir / Path(rel_path).name
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve {rel_path} in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals():
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))

frame_ids = np.array([extract_frame_id(r["image_path"]) for r in manifest_rows])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

for seg in seg_to_indices:
    seg_to_indices[seg].sort(key=lambda i: frame_ids[i])

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor, mode: str = "scalar") -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        g_mag = torch.sqrt(gx**2 + gy**2 + 1e-12)
        if mode == "scalar":
            return g_mag
        elif mode == "directional":
            return torch.cat([gx, gy, g_mag], dim=1)
        raise ValueError(f"Unknown mode: {mode}")

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. BACKBONE & LCF HEAD
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 4. LOSS EVALUATOR
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. MANIFOLD-CONSTRAINED DIRECTIONAL FILTER ON S²
# -------------------------------------------------------------------------
def slerp(u0: np.ndarray, u1: np.ndarray, t: float) -> np.ndarray:
    dot = np.clip(np.dot(u0, u1), -1.0, 1.0)
    theta = np.arccos(dot)
    if theta < 1e-4:
        blend = (1.0 - t) * u0 + t * u1
        return blend / (np.linalg.norm(blend) + 1e-7)
    sin_theta = np.sin(theta)
    return (np.sin((1.0 - t) * theta) / sin_theta) * u0 + (np.sin(t * theta) / sin_theta) * u1

class AdaptiveDirectionalFilter:
    def __init__(self, fps: float = 30.0):
        self.fps = fps
        self.omega_max = 180.0 * (np.pi / 180.0)  # Maximum physiological velocity: 180 deg/s

        sigma_degs = np.array([
            22.5, 27.7, 27.3, 33.6,  # Thumb
            14.3, 22.8, 32.8, 34.6,  # Index
            12.7, 22.6, 37.6, 41.1,  # Middle
            13.5, 25.1, 38.9, 43.6,  # Ring
            15.9, 33.0, 44.8, 48.2   # Pinky
        ])
        self.sigma_R = sigma_degs * (np.pi / 180.0)
        self.R_var = self.sigma_R ** 2

        self.q_rate = np.array([
            0.15, 0.20, 0.25, 0.30,  # Thumb
            0.10, 0.20, 0.35, 0.45,  # Index
            0.08, 0.20, 0.35, 0.45,  # Middle
            0.10, 0.20, 0.35, 0.45,  # Ring
            0.12, 0.25, 0.40, 0.50   # Pinky
        ])

    def filter_sequence(self, raw_dirs: np.ndarray, f_ids: np.ndarray):
        T = len(f_ids)
        filtered_dirs = np.zeros_like(raw_dirs)

        u_state = raw_dirs[0].copy()
        P_cov = self.R_var.copy()
        filtered_dirs[0] = u_state

        total_gates = 0
        total_evals = (T - 1) * 20

        for t in range(1, T):
            delta_frame = max(1, f_ids[t] - f_ids[t - 1])
            delta_t = delta_frame / self.fps

            P_pred = P_cov + self.q_rate * delta_t
            z_meas = raw_dirs[t]

            u_next = np.zeros((20, 3))
            for b in range(20):
                u_p = u_state[b]
                u_m = z_meas[b]

                dot = np.clip(np.dot(u_p, u_m), -1.0, 1.0)
                theta_innov = np.arccos(dot)
                theta_gate = self.omega_max * delta_t + 3.0 * self.sigma_R[b]

                if theta_innov > theta_gate:
                    total_gates += 1
                    clamp_ratio = theta_gate / (theta_innov + 1e-7)
                    u_m = slerp(u_p, u_m, clamp_ratio)

                K = P_pred[b] / (P_pred[b] + self.R_var[b])
                u_next[b] = slerp(u_p, u_m, K)
                P_cov[b] = (1.0 - K) * P_pred[b]

            u_state = u_next
            filtered_dirs[t] = u_state

        gate_rate = (total_gates / max(1, total_evals)) * 100.0
        return filtered_dirs, gate_rate

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS (3 HELD-OUT SPLITS)
# -------------------------------------------------------------------------
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_raw_mpjpe = []
results_filt_mpjpe = []
results_raw_distal = []
results_filt_distal = []
results_raw_fast = []
results_filt_fast = []
results_gating = []
results_tau_star = []

filter_engine = AdaptiveDirectionalFilter(fps=CAPTURE_FPS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

print("=" * 95)
print(f" BENCHMARKING PHYSICAL-TIME DIRECTIONAL FILTER (FPS={CAPTURE_FPS}, 3 HELD-OUT SPLITS)")
print("=" * 95)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
    model = LatentFactorizationModel(in_channels=1).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)

            x_batch = gpu_extractor(raw_rgb, mode="scalar")
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss, _, _, _ = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    val_raw_joints, val_filt_joints, val_gt_joints = [], [], []
    val_displacements = []
    split_gate_rates = []

    with torch.no_grad():
        for seg in val_segs:
            s_indices = seg_to_indices[seg]
            f_ids = frame_ids[s_indices]

            s_raw_dirs, s_gt = [], []
            for i in range(0, len(s_indices), BATCH_SIZE):
                b_idx = s_indices[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                x_batch = gpu_extractor(raw_rgb, mode="scalar")
                _, b_dirs = model(x_batch)
                s_raw_dirs.append(b_dirs.cpu().numpy())
                s_gt.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

            raw_dirs_np = np.concatenate(s_raw_dirs, axis=0)
            gt_joints_np = np.concatenate(s_gt, axis=0)

            filt_dirs_np, gate_pct = filter_engine.filter_sequence(raw_dirs_np, f_ids)
            split_gate_rates.append(gate_pct)

            raw_vecs = torch.tensor(raw_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)
            filt_vecs = torch.tensor(filt_dirs_np * L_CANON.cpu().numpy().reshape(1, 20, 1), device=DEVICE)

            raw_j = reconstruct_joints_from_bones(raw_vecs).cpu().numpy()
            filt_j = reconstruct_joints_from_bones(filt_vecs).cpu().numpy()

            val_raw_joints.append(raw_j)
            val_filt_joints.append(filt_j)
            val_gt_joints.append(gt_joints_np)

            disp_seg = [0.0]
            for t in range(len(s_indices) - 1):
                d = np.linalg.norm(gt_joints_np[t + 1] - gt_joints_np[t], axis=-1).mean()
                disp_seg.append(d)
            val_displacements.append(np.array(disp_seg))

    all_raw = np.concatenate(val_raw_joints, axis=0)
    all_filt = np.concatenate(val_filt_joints, axis=0)
    all_gt = np.concatenate(val_gt_joints, axis=0)
    all_disp = np.concatenate(val_displacements, axis=0)

    # 1. Global MPJPE
    raw_mpjpe = np.linalg.norm(all_raw - all_gt, axis=-1).mean()
    filt_mpjpe = np.linalg.norm(all_filt - all_gt, axis=-1).mean()

    # 2. Distal Direction Error
    b_gt = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]
    u_gt = b_gt / (np.linalg.norm(b_gt, axis=-1, keepdims=True) + 1e-7)

    raw_b = all_raw[:, CHILD_INDICES.cpu().numpy()] - all_raw[:, PARENT_INDICES.cpu().numpy()]
    u_raw = raw_b / (np.linalg.norm(raw_b, axis=-1, keepdims=True) + 1e-7)

    filt_b = all_filt[:, CHILD_INDICES.cpu().numpy()] - all_filt[:, PARENT_INDICES.cpu().numpy()]
    u_filt = filt_b / (np.linalg.norm(filt_b, axis=-1, keepdims=True) + 1e-7)

    deg_raw = np.arccos(np.clip(np.sum(u_raw[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)
    deg_filt = np.arccos(np.clip(np.sum(u_filt[:, DISTAL_BONE_INDICES] * u_gt[:, DISTAL_BONE_INDICES], axis=-1), -1, 1)) * (180.0 / np.pi)

    # 3. Fast-Motion Slice (>95th percentile displacement)
    threshold_95 = np.percentile(all_disp, 95)
    fast_mask = all_disp >= threshold_95
    fast_raw_mpjpe = np.linalg.norm(all_raw[fast_mask] - all_gt[fast_mask], axis=-1).mean()
    fast_filt_mpjpe = np.linalg.norm(all_filt[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    # 4. Lag Shift Search: tau* = argmin_tau E[P_filt(t), P_gt(t + tau)]
    tau_shifts = [-2, -1, 0, 1, 2]
    tau_errors = []
    for tau in tau_shifts:
        if tau < 0:
            err = np.linalg.norm(all_filt[:tau] - all_gt[-tau:], axis=-1).mean()
        elif tau > 0:
            err = np.linalg.norm(all_filt[tau:] - all_gt[:-tau], axis=-1).mean()
        else:
            err = filt_mpjpe
        tau_errors.append(err)
    best_tau = tau_shifts[int(np.argmin(tau_errors))]

    results_raw_mpjpe.append(raw_mpjpe)
    results_filt_mpjpe.append(filt_mpjpe)
    results_raw_distal.append(deg_raw.mean())
    results_filt_distal.append(deg_filt.mean())
    results_raw_fast.append(fast_raw_mpjpe)
    results_filt_fast.append(fast_filt_mpjpe)
    results_gating.append(np.mean(split_gate_rates))
    results_tau_star.append(best_tau)

    print(f"  [Seed {seed:<3}] MPJPE: {raw_mpjpe:5.2f}->{filt_mpjpe:5.2f} mm | Distal: {deg_raw.mean():4.1f}°->{deg_filt.mean():4.1f}° | Fast(>95%): {fast_raw_mpjpe:5.2f}->{fast_filt_mpjpe:5.2f} mm | Gated: {np.mean(split_gate_rates):4.1f}% | tau*: {best_tau}")

    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 7. FINAL COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" ADAPTIVE DIRECTIONAL FILTER: MULTI-METRIC PERFORMANCE AUDIT")
print("=" * 105)
print(f"{'Evaluation Metric':<32} | {'Raw Static LCF':<20} | {'Filtered Kinematic':<20} | {'Delta':<12}")
print("-" * 105)

m_raw, s_raw = np.mean(results_raw_mpjpe), np.std(results_raw_mpjpe)
m_flt, s_flt = np.mean(results_filt_mpjpe), np.std(results_filt_mpjpe)
print(f"{'Overall MPJPE (mm)':<32} | {m_raw:5.2f} ± {s_raw:4.2f} mm    | {m_flt:5.2f} ± {s_flt:4.2f} mm    | {m_flt - m_raw:+5.2f} mm")

m_dr, s_dr = np.mean(results_raw_distal), np.std(results_raw_distal)
m_df, s_df = np.mean(results_filt_distal), np.std(results_filt_distal)
print(f"{'Distal Tip Direction Error':<32} | {m_dr:5.2f}° ± {s_dr:4.2f}°     | {m_df:5.2f}° ± {s_df:4.2f}°     | {m_df - m_dr:+5.2f}°")

m_fr, s_fr = np.mean(results_raw_fast), np.std(results_raw_fast)
m_ff, s_ff = np.mean(results_filt_fast), np.std(results_filt_fast)
print(f"{'Fast-Motion MPJPE (>95th %)':<32} | {m_fr:5.2f} ± {s_fr:4.2f} mm    | {m_ff:5.2f} ± {s_ff:4.2f} mm    | {m_ff - m_fr:+5.2f} mm")

print(f"{'Mean Innovation Gating Rate':<32} | {'—':<20} | {np.mean(results_gating):5.2f} %            | —")
print(f"{'Optimal Lag Shift (tau*)':<32} | {'—':<20} | {int(np.median(results_tau_star)):<2} frames            | —")
print("=" * 105)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Parent-Child Bone Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

# -------------------------------------------------------------------------
# 2. FILE MAP & VRAM INGESTION
# -------------------------------------------------------------------------
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))

    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

print("Building file map across filesystem...")
FILE_MAP = build_file_map(DATA_DIR)
print(f"Indexed {len(FILE_MAP):,} candidate files.")

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p

    p1 = base_dir / rel_clean
    if p1.exists(): return p1

    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2

    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3

    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5

    if fname in FILE_MAP:
        return FILE_MAP[fname]

    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []

    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            fid = extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError(f"Failed to load any images! Sample errors encountered:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 3. CAUSAL TEMPORAL LOOKUP TABLE
# -------------------------------------------------------------------------
seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

prev_frame_indices = np.zeros(len(sample_segments), dtype=np.int64)
for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            prev_frame_indices[i] = i  # Causal boundary condition: repeat frame at t=0
        else:
            prev_frame_indices[i] = indices[step_idx - 1]

vram_prev_indices = torch.tensor(prev_frame_indices, dtype=torch.long, device=DEVICE)

# -------------------------------------------------------------------------
# 4. CANONICAL CONSTANTS & GPU SOBEL ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 5. BACKBONE & LCF HEAD (SUPPORTS VARIABLE INPUT CHANNELS)
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class TemporalLCFModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=in_channels)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 6. EVALUATION MODULE & INPUT BUILDER
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

def build_input_tensor(b_idx: torch.Tensor, mode: str) -> torch.Tensor:
    raw_rgb_t = vram_images[b_idx].float()
    g_t = gpu_extractor(raw_rgb_t)

    if mode == "static_1ch":
        return g_t

    prev_b_idx = vram_prev_indices[b_idx]
    raw_rgb_prev = vram_images[prev_b_idx].float()
    g_prev = gpu_extractor(raw_rgb_prev)

    if mode == "context_2ch":
        return torch.cat([g_t, g_prev], dim=1)
    elif mode == "motion_3ch":
        delta_g = g_t - g_prev
        return torch.cat([g_t, g_prev, delta_g], dim=1)
    raise ValueError(f"Unknown mode: {mode}")

# -------------------------------------------------------------------------
# 7. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
CONFIGURATIONS = [
    ("Static Baseline (|G_t|)",               1, "static_1ch"),
    ("Dual-Frame Context ([|G_t|, |G_t-1|])",  2, "context_2ch"),
    ("Differential Motion ([|G_t|, |G_t-1|, Δ|G|])", 3, "motion_3ch")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _, _ in CONFIGURATIONS}
results_distal_deg = {name: [] for name, _, _ in CONFIGURATIONS}
results_fast_motion = {name: [] for name, _, _ in CONFIGURATIONS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _, _ in CONFIGURATIONS}

print("\n" + "=" * 105)
print(" BENCHMARKING TEMPORAL PERCEPTION (EARLY FUSION) ACROSS 3 HELD-OUT SPLITS")
print("=" * 105)

for exp_name, in_ch, mode in CONFIGURATIONS:
    print(f"\n>>> Running Configuration: {exp_name} ({in_ch} Channels) <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = TemporalLCFModel(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)
                x_batch = build_input_tensor(b_idx, mode)

                optimizer.zero_grad()
                pred_joints, _ = model(x_batch)

                total_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                total_loss.backward()
                optimizer.step()

        # Chronological validation evaluation
        model.eval()
        val_preds = []
        val_targets = []
        val_displacements = []

        with torch.no_grad():
            for seg in val_segs:
                s_indices = seg_to_indices[seg]
                s_idx_tensor = torch.tensor(s_indices, device=DEVICE)

                s_preds = []
                s_gt = []
                for i in range(0, len(s_idx_tensor), BATCH_SIZE):
                    b_idx = s_idx_tensor[i:i + BATCH_SIZE]
                    x_batch = build_input_tensor(b_idx, mode)
                    pred_j, _ = model(x_batch)
                    s_preds.append(pred_j.cpu().numpy())
                    s_gt.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

                s_pred_arr = np.concatenate(s_preds, axis=0)
                s_gt_arr = np.concatenate(s_gt, axis=0)

                val_preds.append(s_pred_arr)
                val_targets.append(s_gt_arr)

                disp_seg = [0.0]
                for t in range(len(s_gt_arr) - 1):
                    d = np.linalg.norm(s_gt_arr[t + 1] - s_gt_arr[t], axis=-1).mean()
                    disp_seg.append(d)
                val_displacements.append(np.array(disp_seg))

        all_pred = np.concatenate(val_preds, axis=0)
        all_gt = np.concatenate(val_targets, axis=0)
        all_disp = np.concatenate(val_displacements, axis=0)

        # 1. MPJPE (mm)
        val_mpjpe = np.linalg.norm(all_pred - all_gt, axis=-1).mean()

        # 2. Per-bone and Distal Direction Error (degrees)
        b_pred = all_pred[:, CHILD_INDICES.cpu().numpy()] - all_pred[:, PARENT_INDICES.cpu().numpy()]
        b_gt = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]

        u_pred = b_pred / (np.linalg.norm(b_pred, axis=-1, keepdims=True) + 1e-7)
        u_gt = b_gt / (np.linalg.norm(b_gt, axis=-1, keepdims=True) + 1e-7)

        cos_sim = np.clip(np.sum(u_pred * u_gt, axis=-1), -1.0, 1.0)
        deg_per_bone = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_per_bone[:, DISTAL_BONE_INDICES].mean()

        # 3. Fast-Motion Slice (>95th percentile displacement)
        threshold_95 = np.percentile(all_disp, 95)
        fast_mask = all_disp >= threshold_95
        fast_mpjpe = np.linalg.norm(all_pred[fast_mask] - all_gt[fast_mask], axis=-1).mean()

        results_mpjpe[exp_name].append(val_mpjpe)
        results_distal_deg[exp_name].append(distal_deg)
        results_fast_motion[exp_name].append(fast_mpjpe)
        accum_bone_deg += deg_per_bone.mean(axis=0)

        print(f"  [Seed {seed:<3}] MPJPE: {val_mpjpe:5.2f} mm | Distal Dir: {distal_deg:4.1f}° | Fast-Motion (>95%): {fast_mpjpe:5.2f} mm")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 8. COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" TEMPORAL PERCEPTION BENCHMARK: 3-SPLIT SEGMENT-DISJOINT EVALUATION")
print("=" * 105)
print(f"{'Input Formulation':<38} | {'MPJPE (mm)':<18} | {'Distal Dir (deg)':<18} | {'Fast-Motion MPJPE':<18}")
print("-" * 105)
for exp_name, _, _ in CONFIGURATIONS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_distal_deg[exp_name]), np.std(results_distal_deg[exp_name])
    m_f, s_f = np.mean(results_fast_motion[exp_name]), np.std(results_fast_motion[exp_name])
    print(f"{exp_name:<38} | {m_p:5.2f} ± {s_p:4.2f} mm    | {m_d:4.1f}° ± {s_d:3.1f}°     | {m_f:5.2f} ± {s_f:4.2f} mm")
print("=" * 105)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: DISTAL STAIRCASE AUDIT (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'Static (|G_t|)':<16} | {'Dual-Frame':<16} | {'Motion Stack':<16} | {'Delta (Mot - Stat)':<18}"
print(header)
print("-" * 105)

deg_stat = per_bone_deg_matrix["Static Baseline (|G_t|)"]
deg_dual = per_bone_deg_matrix["Dual-Frame Context ([|G_t|, |G_t-1|])"]
deg_mot  = per_bone_deg_matrix["Differential Motion ([|G_t|, |G_t-1|, Δ|G|])"]

for k in range(20):
    diff = deg_mot[k] - deg_stat[k]
    sign = "+" if diff > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {deg_stat[k]:5.2f}°          | {deg_dual[k]:5.2f}°          | {deg_mot[k]:5.2f}°          | {sign}{diff:5.2f}°")
print("=" * 105)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 1. FILE MAP & VRAM INGESTION
# -------------------------------------------------------------------------
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

print("Building file map across filesystem...")
FILE_MAP = build_file_map(DATA_DIR)
print(f"Indexed {len(FILE_MAP):,} candidate files.")

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p
    p1 = base_dir / rel_clean
    if p1.exists(): return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5
    if fname in FILE_MAP:
        return FILE_MAP[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None: continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 2. CANONICAL BONE LENGTHS & GPU GRADIENT ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 3. FROZEN PERCEPTION BACKBONE (STATIC LCF)
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=1)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 4. EVALUATION MODULE
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 5. KINEMATIC MANIFOLD PROJECTOR (02_STATE)
# -------------------------------------------------------------------------
class KinematicManifoldProjector:
    """
    Solves: P* = argmin_{P in M} (P - P_meas)^T W (P - P_meas)
    subject to P in M_anatomy.
    """
    def __init__(self, canonical_lens: torch.Tensor):
        self.bone_lens = canonical_lens  # (20,)

        # Empirical measurement standard deviations per bone in degrees
        sigma_degs = torch.tensor([
            22.5, 27.7, 27.3, 33.6,  # Thumb
            14.3, 22.8, 32.8, 34.6,  # Index
            12.7, 22.6, 37.6, 41.1,  # Middle
            13.5, 25.1, 38.9, 43.6,  # Ring
            15.9, 33.0, 44.8, 48.2   # Pinky
        ], device=DEVICE)
        # Confidence weights: w_k = 1 / sigma_k^2 normalized
        self.w_bone = (1.0 / (sigma_degs * (np.pi / 180.0))**2)
        self.w_bone = self.w_bone / self.w_bone.max()  # [0.08 .. 1.0]

    def project_batch(self, raw_dirs: torch.Tensor, stage: str = "raw") -> torch.Tensor:
        """
        raw_dirs: (B, 20, 3) unit bone vectors from LCF
        stage: 'raw' | 'hinge' | 'mcp_bounds' | 'full_coupling'
        Returns: (B, 20, 3) projected unit bone vectors
        """
        if stage == "raw":
            return raw_dirs

        B = raw_dirs.shape[0]
        p_dirs = raw_dirs.clone()

        # Palm Frame from Metacarpals (Middle MCP as longitudinal axis, Index/Pinky span lateral)
        u_mid_meta = p_dirs[:, 8]   # Wrist -> Middle MCP
        u_idx_meta = p_dirs[:, 4]   # Wrist -> Index MCP
        u_pky_meta = p_dirs[:, 16]  # Wrist -> Pinky MCP

        z_palm = u_mid_meta
        trans_palm = F.normalize(u_pky_meta - u_idx_meta, dim=-1)
        y_palm = F.normalize(torch.cross(z_palm, trans_palm, dim=-1), dim=-1) # Palm normal (dorsal/palmar)
        x_palm = F.normalize(torch.cross(y_palm, z_palm, dim=-1), dim=-1)     # Radial/Ulnar lateral

        finger_roots = [4, 8, 12, 16]  # Index, Middle, Ring, Pinky base offsets

        for f_idx, base in enumerate(finger_roots):
            u_meta = p_dirs[:, base]
            u_mcp  = p_dirs[:, base + 1]
            u_pip  = p_dirs[:, base + 2]
            u_dip  = p_dirs[:, base + 3]

            # -------------------------------------------------------------
            # Stage 2+: MCP Abduction & Flexion Bounds
            # -------------------------------------------------------------
            if stage in ["mcp_bounds", "full_coupling"]:
                # Lateral abduction axis: cross(palm_normal, metacarpal_dir)
                flex_axis = F.normalize(torch.cross(y_palm, u_meta, dim=-1), dim=-1)
                abd_axis  = F.normalize(torch.cross(flex_axis, u_meta, dim=-1), dim=-1)

                dot_long = torch.clamp((u_mcp * u_meta).sum(dim=-1, keepdim=True), -1.0, 1.0)
                dot_abd  = (u_mcp * flex_axis).sum(dim=-1, keepdim=True)
                dot_flex = (u_mcp * abd_axis).sum(dim=-1, keepdim=True)

                theta_mcp_flex = torch.atan2(dot_flex, dot_long) # [-10 deg .. 90 deg]
                theta_mcp_abd  = torch.asin(torch.clamp(dot_abd, -0.99, 0.99)) # [-15 deg .. 15 deg]

                # Clamp MCP bounds
                theta_mcp_flex = torch.clamp(theta_mcp_flex, -10.0 * (np.pi / 180.0), 90.0 * (np.pi / 180.0))
                theta_mcp_abd  = torch.clamp(theta_mcp_abd,  -15.0 * (np.pi / 180.0), 15.0 * (np.pi / 180.0))

                # Reconstruct bounded MCP direction
                u_mcp_clamped = (
                    torch.cos(theta_mcp_flex) * torch.cos(theta_mcp_abd) * u_meta +
                    torch.sin(theta_mcp_abd) * flex_axis +
                    torch.sin(theta_mcp_flex) * torch.cos(theta_mcp_abd) * abd_axis
                )
                u_mcp = F.normalize(u_mcp_clamped, dim=-1)
                p_dirs[:, base + 1] = u_mcp

            # -------------------------------------------------------------
            # Stage 1+: Planar Hinge on PIP & DIP (No Hyperextension)
            # -------------------------------------------------------------
            # True anatomical hinge axis of the finger: normal to MCP-PIP plane
            hinge_norm = F.normalize(torch.cross(u_meta, u_mcp, dim=-1), dim=-1)
            # Degeneracy guard: fallback to palm lateral if collinear
            collinear = (torch.norm(torch.cross(u_meta, u_mcp, dim=-1), dim=-1, keepdim=True) < 1e-3)
            hinge_norm = torch.where(collinear, x_palm, hinge_norm)

            # Project PIP into flexion plane perpendicular to hinge normal
            u_pip_planar = u_pip - (u_pip * hinge_norm).sum(dim=-1, keepdim=True) * hinge_norm
            u_pip_planar = F.normalize(u_pip_planar, dim=-1)

            # Compute PIP Flexion Angle
            dot_pip = torch.clamp((u_pip_planar * u_mcp).sum(dim=-1, keepdim=True), -1.0, 1.0)
            cross_pip = (torch.cross(u_mcp, u_pip_planar, dim=-1) * hinge_norm).sum(dim=-1, keepdim=True)
            theta_pip = torch.atan2(cross_pip, dot_pip)

            # Hinge limit: PIP flexion in [0 deg, 105 deg] (no hyperextension)
            theta_pip_clamped = torch.clamp(theta_pip, 0.0, 105.0 * (np.pi / 180.0))

            # Reconstruct PIP vector
            perp_pip = F.normalize(torch.cross(hinge_norm, u_mcp, dim=-1), dim=-1)
            u_pip_proj = torch.cos(theta_pip_clamped) * u_mcp + torch.sin(theta_pip_clamped) * perp_pip
            u_pip_final = F.normalize(u_pip_proj, dim=-1)
            p_dirs[:, base + 2] = u_pip_final

            # Project DIP into same planar flexion hinge
            u_dip_planar = u_dip - (u_dip * hinge_norm).sum(dim=-1, keepdim=True) * hinge_norm
            u_dip_planar = F.normalize(u_dip_planar, dim=-1)

            dot_dip = torch.clamp((u_dip_planar * u_pip_final).sum(dim=-1, keepdim=True), -1.0, 1.0)
            cross_dip = (torch.cross(u_pip_final, u_dip_planar, dim=-1) * hinge_norm).sum(dim=-1, keepdim=True)
            theta_dip = torch.atan2(cross_dip, dot_dip)

            # -------------------------------------------------------------
            # Stage 3+: Anatomical DIP-PIP Tendon Coupling (Landsmeer)
            # -------------------------------------------------------------
            if stage == "full_coupling":
                # Extensor coupling: theta_DIP in [0.5 * theta_PIP - 10 deg, 0.85 * theta_PIP + 10 deg]
                min_dip = torch.clamp(0.50 * theta_pip_clamped - 10.0 * (np.pi / 180.0), min=0.0)
                max_dip = torch.clamp(0.85 * theta_pip_clamped + 10.0 * (np.pi / 180.0), max=85.0 * (np.pi / 180.0))
                theta_dip_clamped = torch.clamp(theta_dip, min_dip, max_dip)
            else:
                # Stage 1 & 2 Hinge limit: DIP flexion in [0 deg, 85 deg]
                theta_dip_clamped = torch.clamp(theta_dip, 0.0, 85.0 * (np.pi / 180.0))

            perp_dip = F.normalize(torch.cross(hinge_norm, u_pip_final, dim=-1), dim=-1)
            u_dip_proj = torch.cos(theta_dip_clamped) * u_pip_final + torch.sin(theta_dip_clamped) * perp_dip
            p_dirs[:, base + 3] = F.normalize(u_dip_proj, dim=-1)

        # Thumb Articulation: Hinge on IP, bounded CMC/MCP
        if stage in ["hinge", "mcp_bounds", "full_coupling"]:
            u_th_mcp = p_dirs[:, 1]
            u_th_ip  = p_dirs[:, 2]
            u_th_tip = p_dirs[:, 3]

            th_hinge = F.normalize(torch.cross(u_th_mcp, u_th_ip, dim=-1), dim=-1)
            u_tip_planar = u_th_tip - (u_th_tip * th_hinge).sum(dim=-1, keepdim=True) * th_hinge
            u_tip_planar = F.normalize(u_tip_planar, dim=-1)

            dot_th = torch.clamp((u_tip_planar * u_th_ip).sum(dim=-1, keepdim=True), -1.0, 1.0)
            cross_th = (torch.cross(u_th_ip, u_tip_planar, dim=-1) * th_hinge).sum(dim=-1, keepdim=True)
            theta_th = torch.atan2(cross_th, dot_th)
            theta_th_clamped = torch.clamp(theta_th, 0.0, 80.0 * (np.pi / 180.0)) # Non-negative IP flexion

            perp_th = F.normalize(torch.cross(th_hinge, u_th_ip, dim=-1), dim=-1)
            p_dirs[:, 3] = F.normalize(torch.cos(theta_th_clamped) * u_th_ip + torch.sin(theta_th_clamped) * perp_th, dim=-1)

        return p_dirs

# -------------------------------------------------------------------------
# 6. BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
STAGES = [
    ("Stage 0: Raw Static LCF",          "raw"),
    ("Stage 1: Projection A (+ Hinge)",  "hinge"),
    ("Stage 2: Projection B (+ MCP)",    "mcp_bounds"),
    ("Stage 3: Projection C (+ Coupling)", "full_coupling")
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in STAGES}
results_distal_deg = {name: [] for name, _ in STAGES}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in STAGES}

projector = KinematicManifoldProjector(L_CANON)

print("\n" + "=" * 105)
print(" BENCHMARKING KINEMATIC MANIFOLD PROJECTION (02_STATE) ACROSS 3 HELD-OUT SPLITS")
print("=" * 105)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
    val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

    model = LatentFactorizationModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # Train static LCF backbone
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)

            x_batch = gpu_extractor(raw_rgb)
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss, _, _, _ = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    # Cache raw model predictions on validation split
    model.eval()
    cached_raw_dirs = []
    cached_gt_joints = []

    with torch.no_grad():
        for i in range(0, len(val_indices), BATCH_SIZE):
            b_idx = val_indices[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            x_batch = gpu_extractor(raw_rgb)
            _, b_dirs = model(x_batch)
            cached_raw_dirs.append(b_dirs)
            cached_gt_joints.append(vram_targets[b_idx].view(-1, 21, 3))

    all_raw_dirs = torch.cat(cached_raw_dirs, dim=0)
    all_gt_joints = torch.cat(cached_gt_joints, dim=0)

    # Evaluate each projection stage
    for stage_name, stage_key in STAGES:
        proj_dirs = projector.project_batch(all_raw_dirs, stage=stage_key)
        proj_vecs = proj_dirs * L_CANON.view(1, 20, 1)
        proj_joints = reconstruct_joints_from_bones(proj_vecs)

        # 1. MPJPE (mm)
        mpjpe = torch.norm(proj_joints - all_gt_joints, dim=-1).mean().item()

        # 2. Per-Bone Angular Error (degrees)
        b_pred = proj_joints[:, CHILD_INDICES] - proj_joints[:, PARENT_INDICES]
        b_gt   = all_gt_joints[:, CHILD_INDICES] - all_gt_joints[:, PARENT_INDICES]

        u_p = b_pred / (torch.norm(b_pred, dim=-1, keepdim=True) + 1e-7)
        u_g = b_gt / (torch.norm(b_gt, dim=-1, keepdim=True) + 1e-7)

        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        deg_bones = torch.acos(cos_sim) * (180.0 / np.pi)

        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean().item()

        results_mpjpe[stage_name].append(mpjpe)
        results_distal_deg[stage_name].append(distal_deg)
        per_bone_deg_matrix[stage_name] += deg_bones.mean(dim=0).cpu().numpy()

        print(f"  [Seed {seed:<3}] {stage_name:<34} | MPJPE: {mpjpe:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

    del model, optimizer
    torch.cuda.empty_cache()

for name, _ in STAGES:
    per_bone_deg_matrix[name] /= len(SEEDS)

# -------------------------------------------------------------------------
# 7. COMPARATIVE AUDIT REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 90)
print(" KINEMATIC MANIFOLD PROJECTION BENCHMARK: 3-SPLIT SUMMARY")
print("=" * 90)
print(f"{'Projection Stage':<38} | {'MPJPE (mm)':<18} | {'Distal Dir (deg)':<18}")
print("-" * 90)
for stage_name, _ in STAGES:
    m_p, s_p = np.mean(results_mpjpe[stage_name]), np.std(results_mpjpe[stage_name])
    m_d, s_d = np.mean(results_distal_deg[stage_name]), np.std(results_distal_deg[stage_name])
    print(f"{stage_name:<38} | {m_p:5.2f} ± {s_p:4.2f} mm    | {m_d:4.1f}° ± {s_d:3.1f}°")
print("=" * 90)

print("\n" + "=" * 115)
print(" PER-BONE DIRECTION ERROR: MANIFOLD ABLATION BREAKDOWN (DEGREES)")
print("=" * 115)
header = f"{'Bone Identifier':<24} | {'Raw Static':<12} | {'+ Hinge':<12} | {'+ MCP':<12} | {'+ Coupling':<12} | {'Net Delta':<10}"
print(header)
print("-" * 115)

d_raw = per_bone_deg_matrix["Stage 0: Raw Static LCF"]
d_hng = per_bone_deg_matrix["Stage 1: Projection A (+ Hinge)"]
d_mcp = per_bone_deg_matrix["Stage 2: Projection B (+ MCP)"]
d_cpl = per_bone_deg_matrix["Stage 3: Projection C (+ Coupling)"]

for k in range(20):
    delta = d_cpl[k] - d_raw[k]
    sign = "+" if delta > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {d_raw[k]:5.2f}°      | {d_hng[k]:5.2f}°      | {d_mcp[k]:5.2f}°      | {d_cpl[k]:5.2f}°      | {sign}{delta:5.2f}°")
print("=" * 115)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------------------------------------------------
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# -------------------------------------------------------------------------
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator : {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192

# 21-Joint Topology (Re:InterHand 24 -> Kinematic 21)
KEEP_21_INDICES = [
    0,               # Wrist
    2, 3, 4, 5,      # Thumb: CMC, MCP, IP, Tip
    6, 7, 8, 9,      # Index: MCP, PIP, DIP, Tip
    10, 11, 12, 13,  # Middle: MCP, PIP, DIP, Tip
    14, 15, 16, 17,  # Ring: MCP, PIP, DIP, Tip
    19, 20, 21, 22   # Pinky: MCP, PIP, DIP, Tip
]
NUM_JOINTS = len(KEEP_21_INDICES)

# 20 Directed Parent-Child Bone Pairs
BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),      # Index
    (0, 9), (9, 10), (10, 11), (11, 12),  # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),# Ring
    (0, 17), (17, 18), (18, 19), (19, 20) # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
NUM_BONES = len(BONE_PAIRS)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# -------------------------------------------------------------------------
# 2. BULLETPROOF FILE MAP & VRAM INGESTION
# -------------------------------------------------------------------------
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

print("Building file map across filesystem...")
FILE_MAP = build_file_map(DATA_DIR)
print(f"Indexed {len(FILE_MAP):,} candidate files.")

def resolve_file(base_dir: Path, rel_path: str, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p
    p1 = base_dir / rel_clean
    if p1.exists(): return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5
    if fname in FILE_MAP:
        return FILE_MAP[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs = [], [], []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None: continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
        except Exception:
            continue

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)
    del raw_imgs, raw_targs, raw_segs
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

# -------------------------------------------------------------------------
# 3. CANONICAL BONE LENGTHS & GPU SOBEL ENGINE
# -------------------------------------------------------------------------
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device)
        self.kx = kx.view(1, 1, 3, 3)
        self.ky = ky.view(1, 1, 3, 3)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx**2 + gy**2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# -------------------------------------------------------------------------
# 4. LCF PERCEPTION BACKBONE
# -------------------------------------------------------------------------
class BaseEncoder(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        feat = self.encoder(x)
        return self.shared_fc(torch.flatten(feat, 1))

class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BaseEncoder(in_channels=1)
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        B = x.shape[0]
        feat = self.backbone(x)
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1 : b_off + 4] = phalanx_cam_u[:, p_off : p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

# -------------------------------------------------------------------------
# 5. DIFFERENTIABLE SOFT ANATOMICAL REGULARIZER
# -------------------------------------------------------------------------
def compute_anatomy_loss(p_dirs: torch.Tensor) -> torch.Tensor:
    """
    Computes smooth soft penalties on anatomical invalidity:
    - Planar hinge deviation on PIP & DIP
    - Hyperextension violations on PIP, DIP, & Thumb IP (with natural tolerance)
    - Excessive MCP knuckle abduction
    - Landsmeer DIP-PIP extensor coupling deviations
    """
    # Palm frame derived from predicted metacarpals
    u_mid_meta = p_dirs[:, 8]
    u_idx_meta = p_dirs[:, 4]
    u_pky_meta = p_dirs[:, 16]

    z_palm = u_mid_meta
    trans_palm = F.normalize(u_pky_meta - u_idx_meta, dim=-1, eps=1e-7)
    y_palm = F.normalize(torch.cross(z_palm, trans_palm, dim=-1), dim=-1) # Palmar normal

    finger_roots = [4, 8, 12, 16] # Index, Middle, Ring, Pinky
    loss_planar = 0.0
    loss_hyperext = 0.0
    loss_coupling = 0.0
    loss_mcp = 0.0

    # Margins and bounds (radians)
    hyperext_margin = 5.0 * (np.pi / 180.0) # 5 deg natural play allowed
    max_pip = 105.0 * (np.pi / 180.0)
    max_dip = 85.0 * (np.pi / 180.0)
    max_mcp_abd = np.sin(18.0 * (np.pi / 180.0))

    for base in finger_roots:
        u_meta = p_dirs[:, base]
        u_mcp  = p_dirs[:, base + 1]
        u_pip  = p_dirs[:, base + 2]
        u_dip  = p_dirs[:, base + 3]

        # Lateral hinge axis running across the finger knuckle
        x_finger = F.normalize(torch.cross(u_meta, y_palm, dim=-1), dim=-1)
        v_flex_mcp = F.normalize(torch.cross(x_finger, u_mcp, dim=-1), dim=-1)
        v_flex_pip = F.normalize(torch.cross(x_finger, u_pip, dim=-1), dim=-1)

        # 1. Hinge Planarity: Lateral deviation out of the flexion plane
        lat_pip = (u_pip * x_finger).sum(dim=-1)
        lat_dip = (u_dip * x_finger).sum(dim=-1)
        loss_planar += (lat_pip.pow(2) + lat_dip.pow(2)).mean()

        # 2. MCP Abduction Bound
        abd_mcp = (u_mcp * x_finger).sum(dim=-1).abs()
        loss_mcp += F.relu(abd_mcp - max_mcp_abd).pow(2).mean()

        # 3. Flexion Angle & Hyperextension Penalties
        sin_pip = (u_pip * v_flex_mcp).sum(dim=-1)
        cos_pip = (u_pip * u_mcp).sum(dim=-1)
        theta_pip = torch.atan2(sin_pip, cos_pip + 1e-7)

        sin_dip = (u_dip * v_flex_pip).sum(dim=-1)
        cos_dip = (u_dip * u_pip).sum(dim=-1)
        theta_dip = torch.atan2(sin_dip, cos_dip + 1e-7)

        loss_hyperext += F.relu(-theta_pip - hyperext_margin).pow(2).mean()
        loss_hyperext += F.relu(theta_pip - max_pip).pow(2).mean()
        loss_hyperext += F.relu(-theta_dip - hyperext_margin).pow(2).mean()
        loss_hyperext += F.relu(theta_dip - max_dip).pow(2).mean()

        # 4. Tendon Extensor Coupling (Landsmeer)
        min_dip = 0.50 * F.relu(theta_pip) - 0.20
        max_dip_cpl = 0.85 * F.relu(theta_pip) + 0.25
        loss_coupling += F.relu(min_dip - theta_dip).pow(2).mean()
        loss_coupling += F.relu(theta_dip - max_dip_cpl).pow(2).mean()

    # Thumb IP hyperextension penalty
    u_th_mcp = p_dirs[:, 1]
    u_th_ip  = p_dirs[:, 2]
    u_th_tip = p_dirs[:, 3]
    th_hinge = F.normalize(torch.cross(u_th_mcp, u_th_ip, dim=-1), dim=-1)
    v_th_flex = F.normalize(torch.cross(th_hinge, u_th_ip, dim=-1), dim=-1)
    sin_th = (u_th_tip * v_th_flex).sum(dim=-1)
    cos_th = (u_th_tip * u_th_ip).sum(dim=-1)
    theta_th = torch.atan2(sin_th, cos_th + 1e-7)
    loss_hyperext += F.relu(-theta_th - hyperext_margin).pow(2).mean()

    total_anat = (loss_planar + loss_hyperext + 0.5 * loss_mcp + 0.5 * loss_coupling) / (len(finger_roots) + 1)
    return total_anat

# -------------------------------------------------------------------------
# 6. LOSS EVALUATOR
# -------------------------------------------------------------------------
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt   = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]

        len_pred = torch.norm(b_pred, dim=-1)
        len_gt   = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)

        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()

        deg_err = torch.acos(cos_sim).mean() * (180.0 / np.pi)
        base_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return base_loss, l_pos.item(), l_len.item(), deg_err.item()

eval_engine = MetricLossEvaluator()

# -------------------------------------------------------------------------
# 7. 3-WAY BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# -------------------------------------------------------------------------
CONFIGURATIONS = [
    ("LCF Baseline (λ_anat = 0.0)",        0.0),
    ("LCF + Weak Anatomy (λ_anat = 5.0)",   5.0),
    ("LCF + Moderate Anatomy (λ_anat = 20.0)", 20.0)
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

results_mpjpe = {name: [] for name, _ in CONFIGURATIONS}
results_distal_deg = {name: [] for name, _ in CONFIGURATIONS}
results_anat_score = {name: [] for name, _ in CONFIGURATIONS}
per_bone_deg_matrix = {name: np.zeros(20) for name, _ in CONFIGURATIONS}

print("\n" + "=" * 105)
print(" BENCHMARKING SOFT ANATOMICAL LATENT REGULARIZATION (3 SEEDS, SEGMENT-DISJOINT)")
print("=" * 105)

for exp_name, lambda_anat in CONFIGURATIONS:
    print(f"\n>>> Running Configuration: {exp_name} <<<")
    accum_bone_deg = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = LatentFactorizationModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]

            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)
                optimizer.zero_grad()
                pred_joints, pred_dirs = model(x_batch)

                base_loss, _, _, _ = eval_engine(pred_joints, y_batch)
                if lambda_anat > 0.0:
                    l_anat = compute_anatomy_loss(pred_dirs)
                    total_loss = base_loss + lambda_anat * l_anat
                else:
                    total_loss = base_loss

                total_loss.backward()
                optimizer.step()

        # Validation evaluation
        model.eval()
        cached_preds = []
        cached_targets = []
        val_anat_scores = []

        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                y_batch = vram_targets[b_idx].view(-1, NUM_JOINTS, 3)

                x_batch = gpu_extractor(raw_rgb)
                pred_joints, pred_dirs = model(x_batch)

                cached_preds.append(pred_joints)
                cached_targets.append(y_batch)
                val_anat_scores.append(compute_anatomy_loss(pred_dirs).item())

        all_pred = torch.cat(cached_preds, dim=0)
        all_gt = torch.cat(cached_targets, dim=0)

        # 1. MPJPE (mm)
        val_mpjpe = torch.norm(all_pred - all_gt, dim=-1).mean().item()

        # 2. Per-Bone Direction Error (degrees)
        b_pred = all_pred[:, CHILD_INDICES] - all_pred[:, PARENT_INDICES]
        b_gt   = all_gt[:, CHILD_INDICES] - all_gt[:, PARENT_INDICES]

        u_pred = b_pred / (torch.norm(b_pred, dim=-1, keepdim=True) + 1e-7)
        u_gt   = b_gt / (torch.norm(b_gt, dim=-1, keepdim=True) + 1e-7)

        cos_sim = torch.clamp((u_pred * u_gt).sum(dim=-1), -1.0, 1.0)
        deg_per_bone = torch.acos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_per_bone[:, DISTAL_BONE_INDICES].mean().item()
        mean_anat_score = np.mean(val_anat_scores)

        results_mpjpe[exp_name].append(val_mpjpe)
        results_distal_deg[exp_name].append(distal_deg)
        results_anat_score[exp_name].append(mean_anat_score)
        accum_bone_deg += deg_per_bone.mean(dim=0).cpu().numpy()

        print(f"  [Seed {seed:<3}] MPJPE: {val_mpjpe:5.2f} mm | Distal Dir: {distal_deg:4.1f}° | Anat Violation Score: {mean_anat_score:6.4f}")

    per_bone_deg_matrix[exp_name] = accum_bone_deg / len(SEEDS)
    del model, optimizer
    torch.cuda.empty_cache()

# -------------------------------------------------------------------------
# 8. COMPARATIVE REPORT
# -------------------------------------------------------------------------
print("\n" + "=" * 105)
print(" SOFT ANATOMICAL REGULARIZATION: 3-SPLIT SEGMENT-DISJOINT SUMMARY")
print("=" * 105)
print(f"{'Configuration':<38} | {'MPJPE (mm)':<18} | {'Distal Dir (deg)':<18} | {'Anat Score':<12}")
print("-" * 105)
for exp_name, _ in CONFIGURATIONS:
    m_p, s_p = np.mean(results_mpjpe[exp_name]), np.std(results_mpjpe[exp_name])
    m_d, s_d = np.mean(results_distal_deg[exp_name]), np.std(results_distal_deg[exp_name])
    m_a, s_a = np.mean(results_anat_score[exp_name]), np.std(results_anat_score[exp_name])
    print(f"{exp_name:<38} | {m_p:5.2f} ± {s_p:4.2f} mm    | {m_d:4.1f}° ± {s_d:3.1f}°     | {m_a:6.4f}")
print("=" * 105)

print("\n" + "=" * 105)
print(" PER-BONE DIRECTION ERROR: SOFT REGULARIZATION BREAKDOWN (DEGREES)")
print("=" * 105)
header = f"{'Bone Identifier':<24} | {'LCF Base (λ=0)':<16} | {'Weak (λ=5)':<16} | {'Mod (λ=20)':<16} | {'Delta (Weak-Base)':<18}"
print(header)
print("-" * 105)

d_base = per_bone_deg_matrix["LCF Baseline (λ_anat = 0.0)"]
d_weak = per_bone_deg_matrix["LCF + Weak Anatomy (λ_anat = 5.0)"]
d_mod  = per_bone_deg_matrix["LCF + Moderate Anatomy (λ_anat = 20.0)"]

for k in range(20):
    diff = d_weak[k] - d_base[k]
    sign = "+" if diff > 0 else ""
    print(f"{BONE_NAMES[k]:<24} | {d_base[k]:5.2f}°          | {d_weak[k]:5.2f}°          | {d_mod[k]:5.2f}°          | {sign}{diff:5.2f}°")
print("=" * 105)

In [ ]:
# @title
import csv
import re
from pathlib import Path
from collections import defaultdict
import numpy as np

# Auto-detect Colab vs. Local hm_ex environment
CANDIDATE_DIRS = [
    Path("/content/dataset"),
    Path("/content"),
    Path("datasets/reinterhand/captures/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands"),
    Path("datasets/reinterhand/captures/m--20210701--1058--0000000--pilot--relightablehandsy--participant0--two-hands/derived/mugsy_subset"),
]

def find_file(filename: str):
    for d in CANDIDATE_DIRS:
        if d.exists():
            p = d / filename
            if p.exists():
                return p
            matches = list(d.rglob(filename))
            if matches:
                return matches[0]
    return None

def extract_fid(s: str) -> int:
    nums = re.findall(r"\d+", str(s))
    return int(nums[-1])

manifest_p = find_file("manifest.csv")
flist_p = find_file("frame_list.txt")
flist_orig_p = find_file("frame_list_orig.txt")

print("=" * 95)
print(" EXPERIMENT A: TIMESTAMP, FRAME-RATE & CONTIGUITY AUDIT")
print("=" * 95)

# 1. Inspect Native Capture Lists (if available locally)
for label, path in [("Raw Capture (frame_list_orig.txt)", flist_orig_p),
                    ("Curated Segments (frame_list.txt)", flist_p)]:
    if path is not None:
        seg_frames = defaultdict(list)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    seg_frames[parts[0]].append(extract_fid(parts[1]))
                elif len(parts) == 1 and parts[0]:
                    seg_frames["__all__"].append(extract_fid(parts[0]))
        deltas = []
        for seg, fids in seg_frames.items():
            fids = sorted(set(fids))
            if len(fids) > 1:
                deltas.extend(np.diff(fids))
        deltas = np.array(deltas)
        unique_d, counts_d = np.unique(deltas, return_counts=True)
        top_strides = ", ".join([f"Δf={d}: {c:,} ({100*c/len(deltas):.1f}%)"
                                 for d, c in zip(unique_d[:5], counts_d[:5])])
        print(f"\n[{label}] ({path})")
        print(f"  Total Segments: {len(seg_frames):,} | Total Transitions: {len(deltas):,}")
        print(f"  Min Δf: {deltas.min()} | Median Δf: {np.median(deltas):.1f} | Mean Δf: {deltas.mean():.2f} | Max Δf: {deltas.max()}")
        print(f"  Dominant Native Strides: {top_strides}")

# 2. Inspect Extracted 5,114-Frame Subset (manifest.csv)
if manifest_p is None:
    raise FileNotFoundError("Could not locate manifest.csv in Colab or local directory.")

seg_to_fids = defaultdict(list)
with open(manifest_p, "r", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        fid = int(r["frame_id"]) if "frame_id" in r else extract_fid(r["image_path"])
        seg_to_fids[r["segment"]].append(fid)

sub_deltas = []
for seg, fids in seg_to_fids.items():
    fids.sort()
    if len(fids) > 1:
        sub_deltas.extend(np.diff(fids))
sub_deltas = np.array(sub_deltas)

print(f"\n[Extracted Visual Subset (manifest.csv)] ({manifest_p})")
print(f"  Total Frames: {sum(len(v) for v in seg_to_fids.values()):,} | Segments: {len(seg_to_fids)} | Transitions: {len(sub_deltas):,}")
print(f"  Min Δf: {sub_deltas.min()} | 25%: {np.percentile(sub_deltas, 25):.0f} | Median: {np.median(sub_deltas):.0f} | 75%: {np.percentile(sub_deltas, 75):.0f} | 95%: {np.percentile(sub_deltas, 95):.0f} | Max: {sub_deltas.max()}")

print("\n  Stride Distribution in manifest.csv:")
for max_df in [1, 2, 3, 6, 9, 15, 30]:
    cnt = int(np.sum(sub_deltas <= max_df))
    print(f"    Transitions with Δf <= {max_df:<2d} : {cnt:4,d} / {len(sub_deltas):,d} ({100.0 * cnt / len(sub_deltas):5.1f}%)")

# 3. Contiguous Run-Length Analysis for Sequence Models (Experiment C)
print("\n  Contiguous Sequence Runs Available (at native stride thresholds):")
for max_gap in [3, 6]:
    run_lengths = []
    for seg, fids in seg_to_fids.items():
        fids = sorted(set(fids))
        cur_len = 1
        for i in range(1, len(fids)):
            if (fids[i] - fids[i - 1]) <= max_gap:
                cur_len += 1
            else:
                run_lengths.append(cur_len)
                cur_len = 1
        run_lengths.append(cur_len)
    runs = np.array(run_lengths)
    w3 = int(np.sum(np.maximum(0, runs - 2)))
    w5 = int(np.sum(np.maximum(0, runs - 4)))
    w8 = int(np.sum(np.maximum(0, runs - 7)))
    print(f"    Max Step Δf <= {max_gap}: Max Run = {runs.max():2d} frames | "
          f"Valid 3-Frame Windows: {w3:4,d} | 5-Frame Windows: {w5:4,d} | 8-Frame Windows: {w8:4,d}")
print("=" * 95)

In [ ]:
# @title
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "Enable GPU in Runtime -> Change runtime type -> T4 GPU"
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

# =============================================================================
# 1. DIFFERENTIABLE / Z-BUFFERED PERSPECTIVE SILHOUETTE & SOBEL RENDERER
# =============================================================================
IMG_H, IMG_W = 128, 128
FOCAL = 380.0
CX, CY = IMG_W / 2.0, IMG_H / 2.0
Z_ROOT = 420.0  # mm from camera pinhole

class GPUSobel(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, img: torch.Tensor) -> torch.Tensor:
        # img: (B, T, H, W) -> returns (B, T, H, W) scalar Sobel magnitude |G|
        B, T, H, W = img.shape
        x = img.view(B * T, 1, H, W)
        gx = F.conv2d(x, self.kx, padding=1) / 4.0
        gy = F.conv2d(x, self.ky, padding=1) / 4.0
        mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)
        return mag.view(B, T, H, W)

sobel_engine = GPUSobel(DEVICE)

# Precompute pixel grid (1, 1, H, W)
yy, xx = torch.meshgrid(
    torch.arange(IMG_H, dtype=torch.float32, device=DEVICE),
    torch.arange(IMG_W, dtype=torch.float32, device=DEVICE),
    indexing="ij"
)
GRID_X = xx.view(1, 1, IMG_H, IMG_W)
GRID_Y = yy.view(1, 1, IMG_H, IMG_W)

@torch.no_grad()
def render_zbuffered_capsules(seg_p0: torch.Tensor, seg_p1: torch.Tensor,
                              radii_mm: torch.Tensor, intensities: torch.Tensor) -> torch.Tensor:
    """
    Renders 3D capsules with perspective projection and Z-buffer occlusion.
    seg_p0, seg_p1: (N, T, S, 3) in camera coordinates (mm, +Z away from camera)
    radii_mm: (S,) physical capsule radii in mm
    intensities: (S,) surface luminance per segment (allows internal occlusion boundaries)
    Returns: (N, T, H, W) rendered grayscale image in [0, 255]
    """
    N, T, S, _ = seg_p0.shape
    canvas = torch.zeros((N, T, IMG_H, IMG_W), dtype=torch.float32, device=DEVICE)
    z_buffer = torch.full((N, T, IMG_H, IMG_W), 1e6, dtype=torch.float32, device=DEVICE)

    for s in range(S):
        p0 = seg_p0[:, :, s, :]  # (N, T, 3)
        p1 = seg_p1[:, :, s, :]  # (N, T, 3)

        z0 = torch.clamp(p0[..., 2:3], min=50.0)
        z1 = torch.clamp(p1[..., 2:3], min=50.0)

        u0 = FOCAL * (p0[..., 0:1] / z0) + CX
        v0 = FOCAL * (p0[..., 1:2] / z0) + CY
        u1 = FOCAL * (p1[..., 0:1] / z1) + CX
        v1 = FOCAL * (p1[..., 1:2] / z1) + CY

        u0 = u0.unsqueeze(-1)
        v0 = v0.unsqueeze(-1)
        u1 = u1.unsqueeze(-1)
        v1 = v1.unsqueeze(-1)
        z0_map = z0.unsqueeze(-1)
        z1_map = z1.unsqueeze(-1)

        du = u1 - u0
        dv = v1 - v0
        len_sq = du ** 2 + dv ** 2 + 1e-6

        t_param = torch.clamp(((GRID_X - u0) * du + (GRID_Y - v0) * dv) / len_sq, 0.0, 1.0)
        proj_u = u0 + t_param * du
        proj_v = v0 + t_param * dv
        dist_px = torch.sqrt((GRID_X - proj_u) ** 2 + (GRID_Y - proj_v) ** 2 + 1e-6)

        z_seg = z0_map + t_param * (z1_map - z0_map)
        rad_px = FOCAL * radii_mm[s] / z_seg

        # Smooth anti-aliased silhouette edge (1.2 px ramp)
        alpha = torch.clamp((rad_px - dist_px) / 1.2 + 0.5, 0.0, 1.0)
        mask = alpha > 0.05

        # Z-buffer occlusion update: closer segments overwrite farther segments
        closer = mask & (z_seg < (z_buffer - 1.5))
        z_buffer = torch.where(closer, z_seg, z_buffer)
        canvas = torch.where(closer, alpha * intensities[s], canvas)

    return canvas

# =============================================================================
# 2. CONJUGATE NECKER-AMBIGUOUS FINGER GENERATOR (T = 5 CONSECUTIVE FRAMES)
# =============================================================================
L_META, L_PROX, L_MID, L_DIST = 48.0, 32.0, 22.0, 17.0
RADII = torch.tensor([8.5, 7.2, 6.2, 5.4, 7.0], device=DEVICE)       # 4 finger bones + 1 occluder digit
INTENSITIES = torch.tensor([190., 205., 220., 235., 165.], device=DEVICE)

@torch.no_grad()
def generate_conjugate_sequences(num_pairs: int, regime: str, seed: int = 42):
    """
    Generates N conjugate pairs (Branch 0: -Z toward camera, Branch 1: +Z away from camera)
    that project to matched 2D coordinates at t=0, and evolves them over T=5 frames.
    """
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    N = num_pairs
    T = 5

    base_x = (torch.rand((N, 1), generator=g, device=DEVICE) - 0.5) * 16.0
    base_y = (torch.rand((N, 1), generator=g, device=DEVICE) - 0.5) * 16.0 + 28.0
    in_plane_roll = (torch.rand((N, 1), generator=g, device=DEVICE) - 0.5) * 0.45  # rad

    # Ambiguous out-of-plane distal tilt magnitude at t=0: [24 deg .. 42 deg]
    theta0 = (24.0 + 18.0 * torch.rand((N, 1), generator=g, device=DEVICE)) * (np.pi / 180.0)

    # Create 2N samples: first N are Branch A (sign = -1, toward camera), next N are Branch B (sign = +1, away)
    signs = torch.cat([-torch.ones((N, 1), device=DEVICE), torch.ones((N, 1), device=DEVICE)], dim=0)
    base_x = base_x.repeat(2, 1)
    base_y = base_y.repeat(2, 1)
    in_plane_roll = in_plane_roll.repeat(2, 1)
    theta0 = theta0.repeat(2, 1)

    t_steps = torch.arange(T, dtype=torch.float32, device=DEVICE).view(1, T)  # [0, 1, 2, 3, 4]

    if regime == "regime1_static":
        theta_t = theta0 + 0.0 * t_steps
        wrist_yaw = torch.zeros((2 * N, T), device=DEVICE)
        occ_offset_x = torch.full((2 * N, T), 999.0, device=DEVICE)
    elif regime == "regime2_articulation":
        omega = (4.5 * np.pi / 180.0)
        theta_t = theta0 + omega * t_steps
        wrist_yaw = torch.zeros((2 * N, T), device=DEVICE)
        occ_offset_x = torch.full((2 * N, T), 999.0, device=DEVICE)
    elif regime == "regime3_occlusion":
        theta_t = theta0 + (1.5 * np.pi / 180.0) * t_steps
        wrist_yaw = torch.zeros((2 * N, T), device=DEVICE)
        occ_offset_x = -18.0 + 9.0 * t_steps.expand(2 * N, T)
    elif regime == "regime4_wrist_rot":
        theta_t = theta0 + (1.0 * np.pi / 180.0) * t_steps
        wrist_yaw = (3.0 * np.pi / 180.0) * t_steps.expand(2 * N, T)
        occ_offset_x = torch.full((2 * N, T), 999.0, device=DEVICE)
    else:
        raise ValueError(f"Unknown regime: {regime}")

    j0 = torch.stack([base_x.expand(-1, T), base_y.expand(-1, T), torch.zeros((2 * N, T), device=DEVICE)], dim=-1)

    dir_x = torch.sin(in_plane_roll).expand(-1, T)
    dir_y = -torch.cos(in_plane_roll).expand(-1, T)

    j1 = j0 + torch.stack([L_META * dir_x, L_META * dir_y, torch.zeros_like(dir_x)], dim=-1)

    ang_prox = signs * (0.35 * theta_t)
    ang_mid = signs * (0.85 * theta_t)
    ang_dist = signs * (1.35 * theta_t)

    j2_local = j1 + torch.stack([
        L_PROX * torch.cos(ang_prox) * dir_x,
        L_PROX * torch.cos(ang_prox) * dir_y,
        L_PROX * torch.sin(ang_prox)
    ], dim=-1)

    j3_local = j2_local + torch.stack([
        L_MID * torch.cos(ang_mid) * dir_x,
        L_MID * torch.cos(ang_mid) * dir_y,
        L_MID * torch.sin(ang_mid)
    ], dim=-1)

    j4_local = j3_local + torch.stack([
        L_DIST * torch.cos(ang_dist) * dir_x,
        L_DIST * torch.cos(ang_dist) * dir_y,
        L_DIST * torch.sin(ang_dist)
    ], dim=-1)

    # Match perspective projection at t=0 so frame 0 has matched 2D coordinates
    for j_tensor in [j2_local, j3_local, j4_local]:
        z_scale_t0 = (Z_ROOT + j_tensor[:, 0:1, 2]) / Z_ROOT
        j_tensor[:, :, 0] = j_tensor[:, :, 0] * z_scale_t0
        j_tensor[:, :, 1] = j_tensor[:, :, 1] * z_scale_t0

    joints = torch.stack([j0, j1, j2_local, j3_local, j4_local], dim=2)  # (2N, T, 5, 3)

    # Apply global wrist yaw rotation around J1 (MCP pivot) if active
    cy = torch.cos(wrist_yaw).unsqueeze(-1)
    sy = torch.sin(wrist_yaw).unsqueeze(-1)
    rel = joints - j1.unsqueeze(2)
    rot_x = rel[..., 0] * cy + rel[..., 2] * sy
    rot_y = rel[..., 1]
    rot_z = -rel[..., 0] * sy + rel[..., 2] * cy
    joints_cam = torch.stack([rot_x, rot_y, rot_z], dim=-1) + j1.unsqueeze(2)
    joints_cam[..., 2] += Z_ROOT

    seg_p0_finger = joints_cam[:, :, 0:4, :]
    seg_p1_finger = joints_cam[:, :, 1:5, :]

    tip_t0_xy = joints_cam[:, 0:1, 4, 0:2].expand(-1, T, -1)
    occ_p0 = torch.stack([tip_t0_xy[..., 0] + occ_offset_x - 12.0, tip_t0_xy[..., 1] - 6.0,
                          torch.full_like(occ_offset_x, Z_ROOT)], dim=-1).unsqueeze(2)
    occ_p1 = torch.stack([tip_t0_xy[..., 0] + occ_offset_x + 12.0, tip_t0_xy[..., 1] + 4.0,
                          torch.full_like(occ_offset_x, Z_ROOT)], dim=-1).unsqueeze(2)

    seg_p0 = torch.cat([seg_p0_finger, occ_p0], dim=2)
    seg_p1 = torch.cat([seg_p1_finger, occ_p1], dim=2)

    imgs = render_zbuffered_capsules(seg_p0, seg_p1, RADII, INTENSITIES)
    sobel_seq = sobel_engine(imgs)  # (2N, T, H, W)

    distal_vec = joints_cam[:, -1, 4, :] - joints_cam[:, -1, 3, :]
    distal_u = F.normalize(distal_vec, dim=-1)
    tip_rel_mm = joints_cam[:, -1, 4, :] - joints_cam[:, -1, 1, :]
    labels = (signs.squeeze(-1) > 0).long()

    return sobel_seq, distal_u, tip_rel_mm, labels

# =============================================================================
# 3. LIGHTWEIGHT TEMPORAL OBSERVATION ENCODER vs. SINGLE-FRAME CNN
# =============================================================================
class TemporalObservationEncoder(nn.Module):
    def __init__(self, use_temporal: bool = True):
        super().__init__()
        self.use_temporal = use_temporal
        self.spatial_cnn = nn.Sequential(
            nn.Conv2d(1, 24, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(24),
            nn.ReLU(inplace=True),
            nn.Conv2d(24, 48, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            nn.Conv2d(48, 96, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(96 * 16, 128),
            nn.ReLU(inplace=True),
        )
        if use_temporal:
            self.gru = nn.GRU(input_size=128, hidden_size=128, num_layers=1, batch_first=True)
        self.dir_head = nn.Linear(128, 3)
        self.tip_head = nn.Linear(128, 3)
        self.branch_head = nn.Linear(128, 2)

    def forward(self, seq_g: torch.Tensor):
        B, T, H, W = seq_g.shape
        if not self.use_temporal:
            feat = self.spatial_cnn(seq_g[:, -1:, :, :])
        else:
            feats = self.spatial_cnn(seq_g.reshape(B * T, 1, H, W)).view(B, T, 128)
            out, _ = self.gru(feats)
            feat = out[:, -1, :]

        u_pred = F.normalize(self.dir_head(feat), dim=-1, eps=1e-7)
        tip_pred = self.tip_head(feat)
        logits = self.branch_head(feat)
        return u_pred, tip_pred, logits

# =============================================================================
# 4. RUN EXPERIMENT B BENCHMARK ACROSS ALL 4 TEMPORAL REGIMES
# =============================================================================
REGIMES = [
    ("1. Static Ambiguous Persistence (No Motion)", "regime1_static"),
    ("2. Active Finger Articulation (Perspective)", "regime2_articulation"),
    ("3. Adjacent Finger Occlusion Sweep (Z-Order)", "regime3_occlusion"),
    ("4. Continuous Wrist Rotation (3 deg/frame)",  "regime4_wrist_rot"),
]

print("\n" + "=" * 108)
print(" EXPERIMENT B: SYNTHETIC AMBIGUOUS FINGER OBSERVABILITY BENCHMARK (CONJUGATE +/-Z BRANCHES)")
print("=" * 108)
print(f"{'Temporal Regime (T=5 Consecutive Frames)':<44} | {'t=0 Diff':<9} | {'Seq Diff':<9} | {'Model':<12} | {'Sign Acc':<9} | {'Distal Err':<11} | {'Tip MPJPE'}")
print("-" * 108)

for title, reg_key in REGIMES:
    X_tr, U_tr, Tip_tr, Y_tr = generate_conjugate_sequences(400, reg_key, seed=42)
    X_val, U_val, Tip_val, Y_val = generate_conjugate_sequences(150, reg_key, seed=999)

    N_val = 150
    branch_A = X_val[:N_val]
    branch_B = X_val[N_val:]
    diff_t0 = torch.norm(branch_A[:, 0] - branch_B[:, 0], dim=(-2, -1)).mean().item()
    diff_seq = torch.norm(branch_A - branch_B, dim=(-2, -1)).mean().item()

    for model_label, is_temp in [("1-Frame CNN", False), ("5-Frame GRU", True)]:
        torch.manual_seed(42)
        model = TemporalObservationEncoder(use_temporal=is_temp).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)

        BATCH = 64
        for epoch in range(20):
            model.train()
            perm = torch.randperm(len(X_tr), device=DEVICE)
            for i in range(0, len(X_tr), BATCH):
                idx = perm[i:i + BATCH]
                opt.zero_grad()
                u_p, tip_p, logits = model(X_tr[idx])
                loss_dir = (1.0 - (u_p * U_tr[idx]).sum(dim=-1)).mean()
                loss_tip = F.l1_loss(tip_p, Tip_tr[idx])
                loss_cls = F.cross_entropy(logits, Y_tr[idx])
                loss = loss_tip + 25.0 * loss_dir + 2.0 * loss_cls
                loss.backward()
                opt.step()

        model.eval()
        with torch.no_grad():
            u_p, tip_p, logits = model(X_val)
            sign_acc = (logits.argmax(dim=-1) == Y_val).float().mean().item() * 100.0
            cos_sim = torch.clamp((u_p * U_val).sum(dim=-1), -1.0, 1.0)
            deg_err = (torch.acos(cos_sim) * (180.0 / np.pi)).mean().item()
            tip_err = torch.norm(tip_p - Tip_val, dim=-1).mean().item()

        row_title = title if not is_temp else ""
        d0_str = f"{diff_t0:6.2f}" if not is_temp else ""
        ds_str = f"{diff_seq:6.2f}" if not is_temp else ""
        print(f"{row_title:<44} | {d0_str:<9} | {ds_str:<9} | {model_label:<12} | {sign_acc:6.1f}%   | {deg_err:6.2f} deg  | {tip_err:6.2f} mm")
    print("-" * 108)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# =============================================================================
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]

# =============================================================================
# 2. BULLETPROOF FILE MAP & SELF-HEALING VRAM INGESTION
# =============================================================================
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists():
        search_dirs.append(base_dir.parent)
    if Path("/content").exists():
        search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists():
        return p
    p1 = base_dir / rel_clean
    if p1.exists():
        return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists():
        return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists():
            return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists():
            return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists():
            return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")
    if 'sample_fids' not in globals() or len(sample_fids) != len(vram_images):
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest_rows = list(csv.DictReader(f))
        sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                                for r in manifest_rows[:len(vram_images)]])

# =============================================================================
# 3. CHRONOLOGICAL CONTIGUITY INDEXING (df <= 6 GATED)
# =============================================================================
seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

N_SAMPLES = len(vram_images)
idx_t_minus_1 = np.zeros(N_SAMPLES, dtype=np.int64)
idx_t_minus_2 = np.zeros(N_SAMPLES, dtype=np.int64)
delta_f_1 = np.zeros(N_SAMPLES, dtype=np.float32)
delta_f_2 = np.zeros(N_SAMPLES, dtype=np.float32)

MAX_CONTIGUOUS_DF = 6  # Native 1-step (df=3) or 2-step (df=6)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            idx_t_minus_1[i] = i
            idx_t_minus_2[i] = i
            delta_f_1[i] = 0.0
            delta_f_2[i] = 0.0
        else:
            prev1 = indices[step_idx - 1]
            df1 = float(sample_fids[i] - sample_fids[prev1])
            if df1 <= MAX_CONTIGUOUS_DF:
                idx_t_minus_1[i] = prev1
                delta_f_1[i] = df1
                if step_idx >= 2:
                    prev2 = indices[step_idx - 2]
                    df2 = float(sample_fids[prev1] - sample_fids[prev2])
                    if df2 <= MAX_CONTIGUOUS_DF:
                        idx_t_minus_2[i] = prev2
                        delta_f_2[i] = df2
                    else:
                        idx_t_minus_2[i] = prev1
                        delta_f_2[i] = 0.0
                else:
                    idx_t_minus_2[i] = prev1
                    delta_f_2[i] = 0.0
            else:
                # Non-contiguous extraction jump: clamp history to self so stale frames never pollute
                idx_t_minus_1[i] = i
                idx_t_minus_2[i] = i
                delta_f_1[i] = 999.0
                delta_f_2[i] = 999.0

vram_prev1 = torch.tensor(idx_t_minus_1, dtype=torch.long, device=DEVICE)
vram_prev2 = torch.tensor(idx_t_minus_2, dtype=torch.long, device=DEVICE)

dt1_norm = np.where(delta_f_1 <= MAX_CONTIGUOUS_DF, delta_f_1 / 3.0, 0.0).astype(np.float32)
dt2_norm = np.where(delta_f_2 <= MAX_CONTIGUOUS_DF, delta_f_2 / 3.0, 0.0).astype(np.float32)
vram_dt1 = torch.tensor(dt1_norm, dtype=torch.float32, device=DEVICE)
vram_dt2 = torch.tensor(dt2_norm, dtype=torch.float32, device=DEVICE)

# =============================================================================
# 4. CANONICAL LENGTHS & GPU SOBEL ENGINE
# =============================================================================
with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# =============================================================================
# 5. STATIC LCF BACKBONE (04_NETWORK) & TEMPORAL OBSERVATION ENCODER
# =============================================================================
class SpatialEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )

    def forward_spatial(self, x: torch.Tensor):
        fmap = self.encoder(x)                         # (B, 256, 6, 8)
        vec = self.shared_fc(torch.flatten(fmap, 1))   # (B, 512)
        return fmap, vec

class LCFHead(nn.Module):
    def __init__(self, in_dim: int = 512):
        super().__init__()
        self.meta_head = nn.Linear(in_dim, 5 * 3)
        self.latent_frame_head = nn.Linear(in_dim, 6)
        self.phalanx_head = nn.Linear(in_dim, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, feat: torch.Tensor):
        B = feat.shape[0]
        meta_cam_u = F.normalize(self.meta_head(feat).view(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=feat.dtype, device=feat.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]

        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

class StaticLCFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SpatialEncoder()
        self.lcf_head = LCFHead(512)

    def forward(self, g_t: torch.Tensor):
        _, vec_t = self.backbone.forward_spatial(g_t)
        return self.lcf_head(vec_t)

class TemporalCorrespondenceEncoder(nn.Module):
    """
    Takes frozen 04_network spatial feature maps & static LCF embedding across
    consecutive window (t-2, t-1, t) + physical dt conditioning, and predicts
    a residual latent refinement before decoding through LCF + L_CANON.
    """
    def __init__(self, frozen_static: StaticLCFModel):
        super().__init__()
        self.backbone = frozen_static.backbone
        self.lcf_head = frozen_static.lcf_head

        for p in self.backbone.parameters():
            p.requires_grad = False

        self.spatial_corr = nn.Sequential(
            nn.Conv2d(256 * 3, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(64 * 6 * 8, 256),
            nn.ReLU(inplace=True),
        )

        self.gru = nn.GRU(input_size=512 + 256 + 1, hidden_size=256, num_layers=1, batch_first=True)

        self.refine_fc = nn.Sequential(
            nn.Linear(512 + 256, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512)
        )
        nn.init.zeros_(self.refine_fc[-1].weight)
        nn.init.zeros_(self.refine_fc[-1].bias)

    def forward(self, g_t2: torch.Tensor, g_t1: torch.Tensor, g_t: torch.Tensor,
                dt2: torch.Tensor, dt1: torch.Tensor):
        with torch.no_grad():
            fmap_2, vec_2 = self.backbone.forward_spatial(g_t2)
            fmap_1, vec_1 = self.backbone.forward_spatial(g_t1)
            fmap_0, vec_0 = self.backbone.forward_spatial(g_t)

        c_2 = self.spatial_corr(torch.cat([fmap_2, fmap_2, torch.zeros_like(fmap_2)], dim=1))
        c_1 = self.spatial_corr(torch.cat([fmap_1, fmap_2, fmap_1 - fmap_2], dim=1))
        c_0 = self.spatial_corr(torch.cat([fmap_0, fmap_1, fmap_0 - fmap_1], dim=1))

        z_2 = torch.cat([vec_2, c_2, torch.zeros_like(dt2)], dim=-1)
        z_1 = torch.cat([vec_1, c_1, dt2], dim=-1)
        z_0 = torch.cat([vec_0, c_0, dt1], dim=-1)

        seq = torch.stack([z_2, z_1, z_0], dim=1)
        gru_out, _ = self.gru(seq)
        h_t = gru_out[:, -1, :]

        has_contig = (dt1 > 0.0).float()
        delta_vec = self.refine_fc(torch.cat([vec_0, h_t], dim=-1)) * has_contig
        vec_refined = vec_0 + delta_vec

        return self.lcf_head(vec_refined)

# =============================================================================
# 6. LOSS & MULTI-METRIC EVALUATOR
# =============================================================================
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        total_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        return total_loss

eval_engine = MetricLossEvaluator()

def get_window_batch(b_idx: torch.Tensor):
    i_t = b_idx
    i_1 = vram_prev1[b_idx]
    i_2 = vram_prev2[b_idx]
    g_t = gpu_extractor(vram_images[i_t].float())
    g_1 = gpu_extractor(vram_images[i_1].float())
    g_2 = gpu_extractor(vram_images[i_2].float())
    dt1 = vram_dt1[b_idx].unsqueeze(-1)
    dt2 = vram_dt2[b_idx].unsqueeze(-1)
    return g_2, g_1, g_t, dt2, dt1

def compute_audit_metrics(all_pred: np.ndarray, all_gt: np.ndarray,
                          all_df: np.ndarray, all_disp: np.ndarray):
    mpjpe_all = np.linalg.norm(all_pred - all_gt, axis=-1).mean()
    contig_mask = (all_df <= 3.5) & (all_df > 0.5)
    mpjpe_df3 = np.linalg.norm(all_pred[contig_mask] - all_gt[contig_mask], axis=-1).mean()
    short_mask = (all_df <= 6.5) & (all_df > 0.5)
    mpjpe_df6 = np.linalg.norm(all_pred[short_mask] - all_gt[short_mask], axis=-1).mean()
    tip_mpjpe = np.linalg.norm(all_pred[:, FINGERTIP_JOINTS] - all_gt[:, FINGERTIP_JOINTS], axis=-1).mean()

    b_p = all_pred[:, CHILD_INDICES.cpu().numpy()] - all_pred[:, PARENT_INDICES.cpu().numpy()]
    b_g = all_gt[:, CHILD_INDICES.cpu().numpy()] - all_gt[:, PARENT_INDICES.cpu().numpy()]
    u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
    u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
    cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
    deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
    distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()
    distal_deg_df3 = deg_bones[contig_mask][:, DISTAL_BONE_INDICES].mean()

    if np.sum(short_mask) > 20:
        thresh_95 = np.percentile(all_disp[short_mask], 95)
        fast_mask = short_mask & (all_disp >= thresh_95)
    else:
        thresh_95 = np.percentile(all_disp, 95)
        fast_mask = all_disp >= thresh_95
    fast_mpjpe = np.linalg.norm(all_pred[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    return {
        "mpjpe_all": mpjpe_all,
        "mpjpe_df3": mpjpe_df3,
        "mpjpe_df6": mpjpe_df6,
        "tip_mpjpe": tip_mpjpe,
        "distal_deg": distal_deg,
        "distal_deg_df3": distal_deg_df3,
        "fast_mpjpe": fast_mpjpe,
    }

# =============================================================================
# 7. RUN EXPERIMENT C ACROSS 3 SEGMENT-DISJOINT SPLITS
# =============================================================================
SEEDS = [42, 101, 777]
STATIC_EPOCHS = 15
TEMPORAL_EPOCHS = 10
BATCH_SIZE = 64

MODELS = ["Static 04_network (Frozen LCF)", "Temporal Correspondence GRU (df<=6 Gated)"]
metrics_log = {m: defaultdict(list) for m in MODELS}

print("\n" + "=" * 112)
print(" EXPERIMENT C: CONSECUTIVE-FRAME TEMPORAL CORRESPONDENCE BENCHMARK (3 HELD-OUT SPLITS)")
print("=" * 112)

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)

    # 1. Train Frozen Static LCF Baseline
    static_model = StaticLCFModel().to(DEVICE)
    opt_static = torch.optim.AdamW(static_model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, STATIC_EPOCHS + 1):
        static_model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            g_t = gpu_extractor(vram_images[b_idx].float())
            y_b = vram_targets[b_idx].view(-1, 21, 3)
            opt_static.zero_grad()
            pred_j, _ = static_model(g_t)
            loss = eval_engine(pred_j, y_b)
            loss.backward()
            opt_static.step()

    static_model.eval()

    # 2. Train Contiguity-Gated Temporal Correspondence Encoder on Contiguous Windows
    temp_model = TemporalCorrespondenceEncoder(static_model).to(DEVICE)
    trainable_params = [p for p in temp_model.parameters() if p.requires_grad]
    opt_temp = torch.optim.AdamW(trainable_params, lr=8e-4, weight_decay=1e-4)

    contig_train_mask = vram_dt1[train_indices] > 0.0
    contig_train_indices = train_indices[contig_train_mask]

    for epoch in range(1, TEMPORAL_EPOCHS + 1):
        temp_model.train()
        perm = contig_train_indices[torch.randperm(len(contig_train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            g_2, g_1, g_t, dt2, dt1 = get_window_batch(b_idx)
            y_b = vram_targets[b_idx].view(-1, 21, 3)
            opt_temp.zero_grad()
            pred_j, _ = temp_model(g_2, g_1, g_t, dt2, dt1)
            loss = eval_engine(pred_j, y_b)
            loss.backward()
            opt_temp.step()

    temp_model.eval()

    # 3. Evaluate Both Models Chronologically on Held-Out Validation Segments
    for m_name, is_temp in [(MODELS[0], False), (MODELS[1], True)]:
        val_preds, val_gts, val_dfs, val_disps = [], [], [], []
        with torch.no_grad():
            for seg in val_segs:
                s_indices = seg_to_indices[seg]
                s_tensor = torch.tensor(s_indices, device=DEVICE)
                s_p, s_g = [], []
                for i in range(0, len(s_tensor), BATCH_SIZE):
                    b_idx = s_tensor[i:i + BATCH_SIZE]
                    if not is_temp:
                        g_t = gpu_extractor(vram_images[b_idx].float())
                        pj, _ = static_model(g_t)
                    else:
                        g_2, g_1, g_t, dt2, dt1 = get_window_batch(b_idx)
                        pj, _ = temp_model(g_2, g_1, g_t, dt2, dt1)
                    s_p.append(pj.cpu().numpy())
                    s_g.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

                s_p_arr = np.concatenate(s_p, axis=0)
                s_g_arr = np.concatenate(s_g, axis=0)
                val_preds.append(s_p_arr)
                val_gts.append(s_g_arr)
                val_dfs.append(delta_f_1[s_indices])

                disp_seg = [0.0]
                for t in range(len(s_g_arr) - 1):
                    disp_seg.append(np.linalg.norm(s_g_arr[t + 1] - s_g_arr[t], axis=-1).mean())
                val_disps.append(np.array(disp_seg))

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)
        all_df = np.concatenate(val_dfs, axis=0)
        all_dp = np.concatenate(val_disps, axis=0)

        res = compute_audit_metrics(all_p, all_g, all_df, all_dp)
        for k, v in res.items():
            metrics_log[m_name][k].append(v)

        print(f"  [Seed {seed:<3}] {m_name:<42} | All: {res['mpjpe_all']:5.2f} mm | "
              f"Adj(df=3): {res['mpjpe_df3']:5.2f} mm | Tips: {res['tip_mpjpe']:5.2f} mm | "
              f"Distal(df=3): {res['distal_deg_df3']:4.1f}° | Fast: {res['fast_mpjpe']:5.2f} mm")

    del static_model, temp_model, opt_static, opt_temp
    torch.cuda.empty_cache()

# =============================================================================
# 8. SUMMARY AUDIT TABLE
# =============================================================================
print("\n" + "=" * 118)
print(" EXPERIMENT C SUMMARY: CONTIGUOUS TEMPORAL CORRESPONDENCE vs. FROZEN STATIC LCF")
print("=" * 118)
print(f"{'Model Architecture':<42} | {'All MPJPE':<14} | {'Adj (df=3) MPJPE':<16} | {'Fingertips':<14} | {'Distal (df=3)':<14} | {'Fast-Motion'}")
print("-" * 118)
for m_name in MODELS:
    d = metrics_log[m_name]
    print(f"{m_name:<42} | "
          f"{np.mean(d['mpjpe_all']):5.2f}±{np.std(d['mpjpe_all']):4.2f} mm | "
          f"{np.mean(d['mpjpe_df3']):5.2f}±{np.std(d['mpjpe_df3']):4.2f} mm   | "
          f"{np.mean(d['tip_mpjpe']):5.2f}±{np.std(d['tip_mpjpe']):4.2f} mm | "
          f"{np.mean(d['distal_deg_df3']):4.1f}°±{np.std(d['distal_deg_df3']):3.1f}°    | "
          f"{np.mean(d['fast_mpjpe']):5.2f}±{np.std(d['fast_mpjpe']):4.2f} mm")
print("=" * 118)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. HARDWARE ACCELERATOR & PATHS
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)

# 2. FILE MAP & VRAM INGESTION (SELF-HEALING)
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists(): search_dirs.append(base_dir.parent)
    if Path("/content").exists(): search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists(): return p
    p1 = base_dir / rel_clean
    if p1.exists(): return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists(): return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists(): return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists(): return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists(): return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None: continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3: errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()
    print(f"Successfully staged {len(vram_images):,} frames into VRAM.")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

print("Ready for training and evaluation pipeline.")

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    raise RuntimeError("VRAM tensors not found! Please run the VRAM ingestion cell first.")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))
sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                        for r in manifest_rows[:len(vram_images)]])

# Build chronological lookup and delta frame intervals
seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

prev_frame_indices = np.zeros(len(sample_segments), dtype=np.int64)
delta_frames = np.zeros(len(sample_segments), dtype=np.float32)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            prev_frame_indices[i] = i
            delta_frames[i] = 3.0
        else:
            prev_idx = indices[step_idx - 1]
            prev_frame_indices[i] = prev_idx
            df = float(sample_fids[i] - sample_fids[prev_idx])
            delta_frames[i] = max(df, 1.0)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

class LatentFactorizationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).view(-1, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(-1, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((x.shape[0], 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

eval_engine = MetricLossEvaluator()

# =============================================================================
# PHYSICAL-TIME S^2 DIRECTIONAL FILTER (02_STATE) - CONTIGUITY & UNCERTAINTY AWARE
# =============================================================================
class PhysicalTimeS2Filter:
    def __init__(self, canonical_lens: torch.Tensor):
        self.bone_lens = canonical_lens
        # Heterogeneous directional uncertainty per bone (radians)
        self.sigma_R = torch.tensor([
            0.39, 0.48, 0.48, 0.58,  # Thumb
            0.25, 0.40, 0.57, 0.60,  # Index
            0.22, 0.40, 0.66, 0.72,  # Middle
            0.24, 0.44, 0.68, 0.76,  # Ring
            0.28, 0.58, 0.78, 0.84   # Pinky
        ], device=DEVICE)
        self.omega_max = 180.0 * (np.pi / 180.0)  # rad/s

    def filter_segment_sequence(self, raw_dirs_seq: torch.Tensor, df_seq: torch.Tensor):
        """
        raw_dirs_seq: (T, 20, 3) unconstrained LCF bone directions along segment
        df_seq: (T,) hardware frame-ID differences (native consecutive step = 3.0)
        """
        T_len = raw_dirs_seq.shape[0]
        filtered_dirs = []
        gates_fired = 0
        total_evals = 0

        prev_u = raw_dirs_seq[0]
        filtered_dirs.append(prev_u)

        for t in range(1, T_len):
            df_val = float(df_seq[t].item())
            curr_u = raw_dirs_seq[t]

            # If extraction gap > 6 frame ticks (>2 native steps), reset state prior to avoid ghost lag
            if df_val > 6.5:
                filtered_dirs.append(curr_u)
                prev_u = curr_u
                continue

            # Physical elapsed time at 30 FPS where native frame-ID stride is 3 ticks (90 ticks/sec)
            dt_t = torch.clamp(df_seq[t] / 90.0, min=1.0 / 30.0)

            # Tighter innovation gate (1.5 * sigma_R + omega_max * dt) to catch Necker flips on S^2
            theta_gate = self.omega_max * dt_t + 1.5 * self.sigma_R  # (20,)

            dot = torch.clamp((prev_u * curr_u).sum(dim=-1), -1.0, 1.0)
            dist = torch.acos(dot)  # (20,)

            gated = dist > theta_gate
            gates_fired += int(gated.sum().item())
            total_evals += 20

            # Bone-dependent Kalman gain: proximal bones (low sigma_R) trust measurement more;
            # noisy distal bones smooth across consecutive steps (df <= 6), unless gated by fast motion
            base_alpha = torch.clamp(dt_t / (dt_t + 0.04 * (self.sigma_R / 0.40)), min=0.35, max=0.92)
            alpha = torch.where(gated, torch.full_like(base_alpha, 0.95), base_alpha).unsqueeze(-1)  # (20, 1)

            # Spherical linear interpolation (SLERP) on S^2
            dist_col = dist.unsqueeze(-1)
            sin_dist = torch.sin(dist_col)
            small_ang = (sin_dist.abs() < 1e-5)

            w1 = torch.where(small_ang, 1.0 - alpha, torch.sin((1.0 - alpha) * dist_col) / (sin_dist + 1e-7))
            w2 = torch.where(small_ang, alpha, torch.sin(alpha * dist_col) / (sin_dist + 1e-7))

            updated_u = F.normalize(w1 * prev_u + w2 * curr_u, dim=-1, eps=1e-7)
            filtered_dirs.append(updated_u)
            prev_u = updated_u

        return torch.stack(filtered_dirs, dim=0), (gates_fired / max(total_evals, 1)) * 100.0

# =============================================================================
# BENCHMARK HARNESS (3 SEEDS, SEGMENT-DISJOINT)
# =============================================================================
SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

print("\n" + "=" * 108)
print(" EXPERIMENT D: UNCERTAINTY & CONTIGUITY-AWARE S^2 STATE FILTER ACROSS 3 HELD-OUT SPLITS")
print("=" * 108)

raw_mpjpe_list, filt_mpjpe_list = [], []
raw_df3_list, filt_df3_list = [], []
raw_distal_list, filt_distal_list = [], []
fast_raw_list, fast_filt_list = [], []
gating_rates = []

for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)

    shuffled_segs = np.random.permutation(unique_segments)
    split_idx = int(0.8 * len(shuffled_segs))
    train_segs = set(shuffled_segs[:split_idx])
    val_segs = set(shuffled_segs[split_idx:])

    train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)

    model = LatentFactorizationModel().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = train_indices[torch.randperm(len(train_indices))]
        for i in range(0, len(perm), BATCH_SIZE):
            b_idx = perm[i:i + BATCH_SIZE]
            raw_rgb = vram_images[b_idx].float()
            y_batch = vram_targets[b_idx].view(-1, 21, 3)
            x_batch = gpu_extractor(raw_rgb)
            optimizer.zero_grad()
            pred_joints, _ = model(x_batch)
            loss = eval_engine(pred_joints, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    filter_engine = PhysicalTimeS2Filter(L_CANON)

    val_raw_preds, val_filt_preds, val_gts, val_disps, val_dfs = [], [], [], [], []
    segment_gating_rates = []

    with torch.no_grad():
        for seg in val_segs:
            s_indices = seg_to_indices[seg]
            s_tensor = torch.tensor(s_indices, device=DEVICE)

            s_preds_j, s_preds_d, s_gts = [], [], []
            for i in range(0, len(s_tensor), BATCH_SIZE):
                b_idx = s_tensor[i:i + BATCH_SIZE]
                raw_rgb = vram_images[b_idx].float()
                x_batch = gpu_extractor(raw_rgb)
                pj, pd = model(x_batch)
                s_preds_j.append(pj)
                s_preds_d.append(pd)
                s_gts.append(vram_targets[b_idx].view(-1, 21, 3))

            seg_j = torch.cat(s_preds_j, dim=0)
            seg_d = torch.cat(s_preds_d, dim=0)
            seg_gt = torch.cat(s_gts, dim=0)

            seg_df = delta_frames[s_indices]
            seg_df_t = torch.tensor(seg_df, dtype=torch.float32, device=DEVICE)

            filt_d, gate_rate = filter_engine.filter_segment_sequence(seg_d, seg_df_t)
            filt_vecs = filt_d * L_CANON.view(1, 20, 1)
            filt_j = reconstruct_joints_from_bones(filt_vecs)

            val_raw_preds.append(seg_j.cpu().numpy())
            val_filt_preds.append(filt_j.cpu().numpy())
            val_gts.append(seg_gt.cpu().numpy())
            val_dfs.append(seg_df)
            segment_gating_rates.append(gate_rate)

            disp_seg = [0.0]
            s_gt_np = seg_gt.cpu().numpy()
            for t in range(len(s_gt_np) - 1):
                disp_seg.append(np.linalg.norm(s_gt_np[t + 1] - s_gt_np[t], axis=-1).mean())
            val_disps.append(np.array(disp_seg))

    all_raw = np.concatenate(val_raw_preds, axis=0)
    all_filt = np.concatenate(val_filt_preds, axis=0)
    all_gt = np.concatenate(val_gts, axis=0)
    all_disp = np.concatenate(val_disps, axis=0)
    all_df = np.concatenate(val_dfs, axis=0)

    mpjpe_raw = np.linalg.norm(all_raw - all_gt, axis=-1).mean()
    mpjpe_filt = np.linalg.norm(all_filt - all_gt, axis=-1).mean()

    df3_mask = (all_df <= 3.5)
    df3_raw = np.linalg.norm(all_raw[df3_mask] - all_gt[df3_mask], axis=-1).mean()
    df3_filt = np.linalg.norm(all_filt[df3_mask] - all_gt[df3_mask], axis=-1).mean()

    def get_distal_deg(pred_j, gt_j):
        b_p = pred_j[:, CHILD_INDICES.cpu().numpy()] - pred_j[:, PARENT_INDICES.cpu().numpy()]
        b_g = gt_j[:, CHILD_INDICES.cpu().numpy()] - gt_j[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        return (np.arccos(cos_sim) * (180.0 / np.pi))[:, DISTAL_BONE_INDICES].mean()

    dist_raw = get_distal_deg(all_raw, all_gt)
    dist_filt = get_distal_deg(all_filt, all_gt)

    thresh_95 = np.percentile(all_disp, 95)
    fast_mask = all_disp >= thresh_95
    fast_raw = np.linalg.norm(all_raw[fast_mask] - all_gt[fast_mask], axis=-1).mean()
    fast_filt = np.linalg.norm(all_filt[fast_mask] - all_gt[fast_mask], axis=-1).mean()

    raw_mpjpe_list.append(mpjpe_raw)
    filt_mpjpe_list.append(mpjpe_filt)
    raw_df3_list.append(df3_raw)
    filt_df3_list.append(df3_filt)
    raw_distal_list.append(dist_raw)
    filt_distal_list.append(dist_filt)
    fast_raw_list.append(fast_raw)
    fast_filt_list.append(fast_filt)
    gating_rates.append(np.mean(segment_gating_rates))

    print(f" [Seed {seed:<3}] All MPJPE: {mpjpe_raw:5.2f} -> {mpjpe_filt:5.2f} mm | "
          f"Adj(df=3): {df3_raw:5.2f} -> {df3_filt:5.2f} mm | "
          f"Fast (>95%): {fast_raw:5.2f} -> {fast_filt:5.2f} mm | Gated: {gating_rates[-1]:5.2f}%")

    del model, optimizer
    torch.cuda.empty_cache()

print("\n" + "=" * 108)
print(" EXPERIMENT D SUMMARY: CONTIGUITY-GATED S^2 FILTER vs. RAW STATIC LCF")
print("=" * 108)
print(f"{'Evaluation Metric':<34} | {'Raw Static LCF':<18} | {'Filtered Kinematic':<18} | {'Delta'}")
print("-" * 108)
print(f"{'Overall MPJPE (mm)':<34} | {np.mean(raw_mpjpe_list):5.2f} ± {np.std(raw_mpjpe_list):4.2f} mm | {np.mean(filt_mpjpe_list):5.2f} ± {np.std(filt_mpjpe_list):4.2f} mm | {np.mean(filt_mpjpe_list) - np.mean(raw_mpjpe_list):+5.2f} mm")
print(f"{'Consecutive (df=3) MPJPE (mm)':<34} | {np.mean(raw_df3_list):5.2f} ± {np.std(raw_df3_list):4.2f} mm | {np.mean(filt_df3_list):5.2f} ± {np.std(filt_df3_list):4.2f} mm | {np.mean(filt_df3_list) - np.mean(raw_df3_list):+5.2f} mm")
print(f"{'Distal Tip Direction Error':<34} | {np.mean(raw_distal_list):4.1f}° ± {np.std(raw_distal_list):3.1f}° | {np.mean(filt_distal_list):4.1f}° ± {np.std(filt_distal_list):3.1f}° | {np.mean(filt_distal_list) - np.mean(raw_distal_list):+4.1f}°")
print(f"{'Fast-Motion MPJPE (>95th %)':<34} | {np.mean(fast_raw_list):5.2f} ± {np.std(fast_raw_list):4.2f} mm | {np.mean(fast_filt_list):5.2f} ± {np.std(fast_filt_list):4.2f} mm | {np.mean(fast_filt_list) - np.mean(fast_raw_list):+5.2f} mm")
print(f"{'Mean Innovation Gating Rate':<34} | {'-':<18} | {np.mean(gating_rates):5.2f}%             | -")
print("=" * 108)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    raise RuntimeError("VRAM tensors not found! Please run the VRAM ingestion cell first.")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest_rows = list(csv.DictReader(f))
sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                        for r in manifest_rows[:len(vram_images)]])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

# Build contiguity-gated previous frame index (only pair if df <= 6, else self)
contig_prev_indices = np.zeros(len(sample_segments), dtype=np.int64)
delta_frames = np.zeros(len(sample_segments), dtype=np.float32)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            contig_prev_indices[i] = i
            delta_frames[i] = 999.0
        else:
            prev_i = indices[step_idx - 1]
            df = float(sample_fids[i] - sample_fids[prev_i])
            delta_frames[i] = df
            if df <= 6.5:
                contig_prev_indices[i] = prev_i
            else:
                contig_prev_indices[i] = i  # Gate out non-adjacent keyframes

vram_contig_prev = torch.tensor(contig_prev_indices, dtype=torch.long, device=DEVICE)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

# =============================================================================
# 1. MULTI-CUE GPU VISUAL FEATURE EXTRACTOR (EXPERIMENT E)
# =============================================================================
class GPUVisualCueExtractor(nn.Module):
    """
    Extracts illumination-invariant interior appearance and crease polarity cues
    alongside Scalar Sobel magnitude |G_t|:
      - |G_t|    : Scalar Sobel edge magnitude (1 Ch)
      - I_norm   : Instance-standardized foreground luminance (shading/depth cue, 1 Ch)
      - Lap(I)   : Signed Laplacian valley/ridge detector (palmar flexion creases vs dorsal surface, 1 Ch)
      - Chrom_RG : Normalized Red-Green chrominance ratio (palm vs dorsal skin tone cue, 1 Ch)
    """
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        # Signed 3x3 Laplacian kernel (valleys/creases vs ridges)
        klap = torch.tensor([[0.,  1., 0.],
                             [1., -4., 1.],
                             [0.,  1., 0.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)
        self.register_buffer("klap", klap)

    @torch.no_grad()
    def extract(self, b_idx: torch.Tensor, mode: str) -> torch.Tensor:
        rgb = vram_images[b_idx].float()  # (B, 3, H, W) in [0, 255]
        r, g, b = rgb[:, 0:1], rgb[:, 1:2], rgb[:, 2:3]
        gray = 0.299 * r + 0.587 * g + 0.114 * b  # (B, 1, H, W)

        # 1. Scalar Sobel |G_t|
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        sobel_mag = torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

        if mode == "1_sobel_only":
            return sobel_mag

        # Per-frame foreground-aware luminance normalization (eliminates global exposure drift)
        mean_g = gray.mean(dim=(-2, -1), keepdim=True)
        std_g = gray.std(dim=(-2, -1), keepdim=True) + 1.0
        i_norm = torch.clamp((gray - mean_g) / (3.0 * std_g), -1.0, 1.0) * 0.5 + 0.5

        if mode == "2_sobel_luminance":
            return torch.cat([sobel_mag, i_norm], dim=1)

        # Signed Crease Laplacian (highlights dark palmar creases & inter-finger occlusion valleys)
        lap = F.conv2d(gray, self.klap, padding=1) / 255.0
        lap_signed = torch.tanh(lap * 3.0)

        if mode == "3_sobel_crease_lap":
            return torch.cat([sobel_mag, lap_signed], dim=1)

        if mode == "4_full_appearance_contig":
            # Include contiguity-gated previous Sobel |G_{t-1}| (equals |G_t| when df > 6)
            prev_idx = vram_contig_prev[b_idx]
            rgb_prev = vram_images[prev_idx].float()
            gray_prev = 0.299 * rgb_prev[:, 0:1] + 0.587 * rgb_prev[:, 1:2] + 0.114 * rgb_prev[:, 2:3]
            gx_p = F.conv2d(gray_prev, self.kx, padding=1) / 1020.0
            gy_p = F.conv2d(gray_prev, self.ky, padding=1) / 1020.0
            sobel_prev = torch.sqrt(gx_p ** 2 + gy_p ** 2 + 1e-12)
            return torch.cat([sobel_mag, i_norm, lap_signed, sobel_prev], dim=1)

        raise ValueError(f"Unknown mode: {mode}")

cue_extractor = GPUVisualCueExtractor(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# =============================================================================
# 2. LCF MODEL WITH VARIABLE INPUT CHANNELS
# =============================================================================
class LatentFactorizationModel(nn.Module):
    def __init__(self, in_channels: int = 1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).view(-1, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(-1, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((x.shape[0], 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs

class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

eval_engine = MetricLossEvaluator()

# =============================================================================
# 3. BENCHMARK HARNESS (EXPERIMENT E: 4 VISUAL REPRESENTATIONS x 3 SPLITS)
# =============================================================================
CONFIGS = [
    ("1. Scalar Sobel Baseline (|G_t|)",               1, "1_sobel_only"),
    ("2. Sobel + Norm Luminance ([|G_t|, I_norm])",    2, "2_sobel_luminance"),
    ("3. Sobel + Crease Polarity ([|G_t|, Lap(I)])",   2, "3_sobel_crease_lap"),
    ("4. Full Interior + Contig ([|G|, I, Lap, G_p])", 4, "4_full_appearance_contig"),
]

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

res_mpjpe = {name: [] for name, _, _ in CONFIGS}
res_tips = {name: [] for name, _, _ in CONFIGS}
res_distal = {name: [] for name, _, _ in CONFIGS}
per_bone_deg = {name: np.zeros(20) for name, _, _ in CONFIGS}

print("\n" + "=" * 110)
print(" EXPERIMENT E: INTERIOR APPEARANCE, CREASE POLARITY & SHADING BENCHMARK (3 HELD-OUT SPLITS)")
print("=" * 110)

for exp_name, in_ch, mode_key in CONFIGS:
    print(f"\n>>> Running {exp_name} ({in_ch} Ch) <<<")
    accum_bones = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = LatentFactorizationModel(in_channels=in_ch).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]
            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                x_batch = cue_extractor.extract(b_idx, mode_key)
                y_batch = vram_targets[b_idx].view(-1, 21, 3)

                optimizer.zero_grad()
                pred_joints, _ = model(x_batch)
                loss = eval_engine(pred_joints, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        val_preds, val_gts = [], []
        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                x_batch = cue_extractor.extract(b_idx, mode_key)
                pj, _ = model(x_batch)
                val_preds.append(pj.cpu().numpy())
                val_gts.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)

        mpjpe = np.linalg.norm(all_p - all_g, axis=-1).mean()
        tip_err = np.linalg.norm(all_p[:, FINGERTIP_JOINTS] - all_g[:, FINGERTIP_JOINTS], axis=-1).mean()

        b_p = all_p[:, CHILD_INDICES.cpu().numpy()] - all_p[:, PARENT_INDICES.cpu().numpy()]
        b_g = all_g[:, CHILD_INDICES.cpu().numpy()] - all_g[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()

        res_mpjpe[exp_name].append(mpjpe)
        res_tips[exp_name].append(tip_err)
        res_distal[exp_name].append(distal_deg)
        accum_bones += deg_bones.mean(axis=0)

        print(f"  [Seed {seed:<3}] MPJPE: {mpjpe:5.2f} mm | Fingertips: {tip_err:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

        del model, optimizer
        torch.cuda.empty_cache()

    per_bone_deg[exp_name] = accum_bones / len(SEEDS)

# =============================================================================
# 4. SUMMARY REPORT & PER-BONE DISTAL AUDIT
# =============================================================================
print("\n" + "=" * 110)
print(" EXPERIMENT E SUMMARY: VISUAL CUES BEYOND SCALAR SOBEL MAGNITUDE (3 HELD-OUT SPLITS)")
print("=" * 110)
print(f"{'Visual Input Representation':<48} | {'Overall MPJPE':<18} | {'Fingertip MPJPE':<18} | {'Distal Dir Err'}")
print("-" * 110)
base_mpjpe = np.mean(res_mpjpe[CONFIGS[0][0]])
for exp_name, _, _ in CONFIGS:
    mp, sp = np.mean(res_mpjpe[exp_name]), np.std(res_mpjpe[exp_name])
    mt, st = np.mean(res_tips[exp_name]), np.std(res_tips[exp_name])
    md, sd = np.mean(res_distal[exp_name]), np.std(res_distal[exp_name])
    print(f"{exp_name:<48} | {mp:5.2f} ± {sp:4.2f} mm   | {mt:5.2f} ± {st:4.2f} mm   | {md:4.1f}° ± {sd:3.1f}°")
print("=" * 110)

print("\n" + "=" * 110)
print(" PER-BONE DIRECTION ERROR BREAKDOWN (DEGREES)")
print("=" * 110)
c0, c1, c2, c3 = [c[0] for c in CONFIGS]
print(f"{'Bone Identifier':<24} | {'1. |G_t|':<14} | {'2. +Luminance':<14} | {'3. +CreaseLap':<14} | {'4. FullStack':<14} | {'Best Delta'}")
print("-" * 110)
for k in range(20):
    v0 = per_bone_deg[c0][k]
    v1 = per_bone_deg[c1][k]
    v2 = per_bone_deg[c2][k]
    v3 = per_bone_deg[c3][k]
    best_alt = min(v1, v2, v3)
    delta = best_alt - v0
    print(f"{BONE_NAMES[k]:<24} | {v0:5.2f}°        | {v1:5.2f}°        | {v2:5.2f}°        | {v3:5.2f}°        | {delta:+5.2f}°")
print("=" * 110)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# =============================================================================
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
BONE_NAMES = [
    "Wrist -> Thumb CMC", "Thumb CMC -> MCP", "Thumb MCP -> IP", "Thumb IP -> Tip",
    "Wrist -> Index MCP", "Index MCP -> PIP", "Index PIP -> DIP", "Index DIP -> Tip",
    "Wrist -> Middle MCP", "Middle MCP -> PIP", "Middle PIP -> DIP", "Middle DIP -> Tip",
    "Wrist -> Ring MCP", "Ring MCP -> PIP", "Ring PIP -> DIP", "Ring DIP -> Tip",
    "Wrist -> Pinky MCP", "Pinky MCP -> PIP", "Pinky PIP -> DIP", "Pinky DIP -> Tip"
]

# =============================================================================
# 2. SELF-HEALING VRAM INGESTION (LOADS IF NOT ALREADY IN VRAM)
# =============================================================================
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists():
        search_dirs.append(base_dir.parent)
    if Path("/content").exists():
        search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists():
        return p
    p1 = base_dir / rel_clean
    if p1.exists():
        return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists():
        return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists():
            return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists():
            return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists():
            return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")
    if 'sample_fids' not in globals() or len(sample_fids) != len(vram_images):
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest_rows = list(csv.DictReader(f))
        sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                                for r in manifest_rows[:len(vram_images)]])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

# =============================================================================
# 3. GPU SOBEL ENGINE & KINEMATIC HELPERS
# =============================================================================
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

# =============================================================================
# 4. BASELINE GLOBAL LCF (6x8 POOLING) vs. HIERARCHICAL SPATIAL-ROI LCF (24x32)
# =============================================================================
class GlobalLCFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def forward(self, x):
        feat = self.shared_fc(torch.flatten(self.encoder(x), 1))
        meta_cam_u = F.normalize(self.meta_head(feat).view(-1, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).view(-1, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((x.shape[0], 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs


class HierarchicalSpatialLCFModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),   # 96 x 128
            nn.BatchNorm2d(32), nn.ReLU(inplace=True)
        )
        self.stage2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),  # 48 x 64
            nn.BatchNorm2d(64), nn.ReLU(inplace=True)
        )
        self.stage3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 24 x 32
            nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.stage4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),# 12 x 16
            nn.BatchNorm2d(256), nn.ReLU(inplace=True)
        )

        self.global_pool = nn.AdaptiveAvgPool2d((6, 8))
        self.global_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)

        self.fpn_fuse = nn.Sequential(
            nn.Conv2d(64 + 128 + 256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )
        self.spatial_attn_conv = nn.Conv2d(128, 15, kernel_size=1)

        self.prox_fc = nn.Sequential(nn.Linear(512 + 130 + 3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))
        self.mid_fc = nn.Sequential(nn.Linear(512 + 130 + 3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))
        self.dist_fc = nn.Sequential(nn.Linear(512 + 130 + 3, 128), nn.ReLU(inplace=True), nn.Linear(128, 3))

        self.log_var_head = nn.Linear(512, 20)
        self.register_buffer("bone_lens", L_CANON)

        yy, xx = torch.meshgrid(
            torch.linspace(-1.0, 1.0, 24, device=DEVICE),
            torch.linspace(-1.0, 1.0, 32, device=DEVICE),
            indexing="ij"
        )
        # Fixed: use .reshape() on non-contiguous meshgrid tensors
        self.register_buffer("grid_x", xx.contiguous().reshape(1, 1, 24 * 32))
        self.register_buffer("grid_y", yy.contiguous().reshape(1, 1, 24 * 32))

    def forward(self, x):
        B = x.shape[0]
        f1 = self.stage1(x)
        f2 = self.stage2(f1)  # (B, 64, 48, 64)
        f3 = self.stage3(f2)  # (B, 128, 24, 32)
        f4 = self.stage4(f3)  # (B, 256, 12, 16)

        g_feat = self.global_fc(torch.flatten(self.global_pool(f4), 1))  # (B, 512)
        meta_cam_u = F.normalize(self.meta_head(g_feat).reshape(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(g_feat)) # (B, 3, 3)

        meta_local_u = torch.bmm(R_latent.transpose(1, 2), meta_cam_u.transpose(1, 2)).transpose(1, 2)

        f2_down = F.avg_pool2d(f2, kernel_size=2, stride=2)
        f4_up = F.interpolate(f4, size=(24, 32), mode="bilinear", align_corners=False)
        fpn = self.fpn_fuse(torch.cat([f2_down, f3, f4_up], dim=1))      # (B, 128, 24, 32)

        attn_logits = self.spatial_attn_conv(fpn).reshape(B, 15, 24 * 32)
        attn_weights = F.softmax(attn_logits, dim=-1)                    # (B, 15, 768)

        fpn_flat = fpn.reshape(B, 128, 24 * 32)
        local_feats = torch.bmm(attn_weights, fpn_flat.transpose(1, 2))  # (B, 15, 128)
        exp_x = (attn_weights * self.grid_x).sum(dim=-1, keepdim=True)
        exp_y = (attn_weights * self.grid_y).sum(dim=-1, keepdim=True)
        local_desc = torch.cat([local_feats, exp_x, exp_y], dim=-1)      # (B, 15, 130)

        local_desc_5x3 = local_desc.reshape(B, 5, 3, 130)
        g_exp = g_feat.unsqueeze(1).expand(B, 5, 512)

        prox_in = torch.cat([g_exp, local_desc_5x3[:, :, 0, :], meta_local_u], dim=-1)
        prox_local_u = F.normalize(self.prox_fc(prox_in), dim=-1, eps=1e-7)

        mid_in = torch.cat([g_exp, local_desc_5x3[:, :, 1, :], prox_local_u], dim=-1)
        mid_local_u = F.normalize(self.mid_fc(mid_in), dim=-1, eps=1e-7)

        dist_in = torch.cat([g_exp, local_desc_5x3[:, :, 2, :], mid_local_u], dim=-1)
        dist_local_u = F.normalize(self.dist_fc(dist_in), dim=-1, eps=1e-7)

        phalanx_local_u = torch.stack([prox_local_u, mid_local_u, dist_local_u], dim=2).reshape(B, 15, 3)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=x.dtype, device=x.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        log_var = torch.clamp(self.log_var_head(g_feat), min=-4.0, max=4.0)
        return joints, bone_dirs, log_var

# =============================================================================
# 5. LOSS EVALUATOR & BENCHMARK HARNESS (3 HELD-OUT SPLITS)
# =============================================================================
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor, log_var: torch.Tensor = None):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        bone_dir_err = 1.0 - cos_sim
        l_dir = bone_dir_err.mean()

        base_loss = l_pos + 0.5 * l_len + 20.0 * l_dir
        if log_var is not None:
            l_unc = (torch.exp(-log_var) * bone_dir_err.detach() + 0.5 * log_var).mean()
            base_loss = base_loss + 0.5 * l_unc
        return base_loss

eval_engine = MetricLossEvaluator()

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

MODELS = [
    "1. Global LCF Baseline (6x8 Pool)",
    "2. Hierarchical Spatial-FPN LCF (24x32 ROI + Chain)",
]

res_mpjpe = {m: [] for m in MODELS}
res_tips = {m: [] for m in MODELS}
res_distal = {m: [] for m in MODELS}
per_bone_deg = {m: np.zeros(20) for m in MODELS}

print("\n" + "=" * 110)
print(" EXPERIMENT F: GLOBAL LCF (6x8) vs. HIERARCHICAL SPATIAL-ROI FPN LCF (24x32) ACROSS 3 SPLITS")
print("=" * 110)

for m_name in MODELS:
    is_hier = "Hierarchical" in m_name
    print(f"\n>>> Running {m_name} <<<")
    accum_bones = np.zeros(20)

    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        model = HierarchicalSpatialLCFModel().to(DEVICE) if is_hier else GlobalLCFModel().to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]
            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                x_batch = gpu_extractor(vram_images[b_idx].float())
                y_batch = vram_targets[b_idx].view(-1, 21, 3)

                optimizer.zero_grad()
                if is_hier:
                    pred_j, _, log_v = model(x_batch)
                    loss = eval_engine(pred_j, y_batch, log_v)
                else:
                    pred_j, _ = model(x_batch)
                    loss = eval_engine(pred_j, y_batch)
                loss.backward()
                optimizer.step()

        model.eval()
        val_preds, val_gts = [], []
        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                x_batch = gpu_extractor(vram_images[b_idx].float())
                out = model(x_batch)
                pj = out[0]
                val_preds.append(pj.cpu().numpy())
                val_gts.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)

        mpjpe = np.linalg.norm(all_p - all_g, axis=-1).mean()
        tip_err = np.linalg.norm(all_p[:, FINGERTIP_JOINTS] - all_g[:, FINGERTIP_JOINTS], axis=-1).mean()

        b_p = all_p[:, CHILD_INDICES.cpu().numpy()] - all_p[:, PARENT_INDICES.cpu().numpy()]
        b_g = all_g[:, CHILD_INDICES.cpu().numpy()] - all_g[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()

        res_mpjpe[m_name].append(mpjpe)
        res_tips[m_name].append(tip_err)
        res_distal[m_name].append(distal_deg)
        accum_bones += deg_bones.mean(axis=0)

        print(f"  [Seed {seed:<3}] MPJPE: {mpjpe:5.2f} mm | Fingertips: {tip_err:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

        del model, optimizer
        torch.cuda.empty_cache()

    per_bone_deg[m_name] = accum_bones / len(SEEDS)

print("\n" + "=" * 110)
print(" EXPERIMENT F SUMMARY: SPATIAL RESOLUTION & HIERARCHICAL CHAIN DECODING")
print("=" * 110)
print(f"{'Architecture':<52} | {'Overall MPJPE':<18} | {'Fingertip MPJPE':<18} | {'Distal Dir Err'}")
print("-" * 110)
for m_name in MODELS:
    mp, sp = np.mean(res_mpjpe[m_name]), np.std(res_mpjpe[m_name])
    mt, st = np.mean(res_tips[m_name]), np.std(res_tips[m_name])
    md, sd = np.mean(res_distal[m_name]), np.std(res_distal[m_name])
    print(f"{m_name:<52} | {mp:5.2f} ± {sp:4.2f} mm   | {mt:5.2f} ± {st:4.2f} mm   | {md:4.1f}° ± {sd:3.1f}°")
print("=" * 110)

print("\n" + "=" * 95)
print(" PER-BONE DIRECTION ERROR BREAKDOWN (DEGREES)")
print("=" * 95)
m0, m1 = MODELS[0], MODELS[1]
print(f"{'Bone Identifier':<24} | {'1. Global (6x8)':<18} | {'2. Hier-FPN (24x32)':<20} | {'Delta'}")
print("-" * 95)
for k in range(20):
    v0 = per_bone_deg[m0][k]
    v1 = per_bone_deg[m1][k]
    print(f"{BONE_NAMES[k]:<24} | {v0:6.2f}°            | {v1:6.2f}°              | {v1 - v0:+5.2f}°")
print("=" * 95)

In [ ]:
# @title
import os
import gc
import csv
import json
import re
from pathlib import Path
from collections import defaultdict
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# =============================================================================
# 1. HARDWARE ACCELERATOR & DATASET PATHS
# =============================================================================
assert torch.cuda.is_available(), "GPU is not enabled! Set Runtime -> Change runtime type -> T4 GPU."
DEVICE = torch.device("cuda")
print(f"Hardware Accelerator: {torch.cuda.get_device_name(0)}")

DATA_DIR = Path("/content/dataset")
MANIFEST_PATH = DATA_DIR / "manifest.csv"
if not MANIFEST_PATH.exists():
    alt_manifest = Path("/content/manifest.csv")
    if alt_manifest.exists():
        MANIFEST_PATH = alt_manifest
        DATA_DIR = Path("/content")
    else:
        raise FileNotFoundError(f"Cannot find manifest.csv at {MANIFEST_PATH} or {alt_manifest}")

TARGET_W, TARGET_H = 256, 192
KEEP_21_INDICES = [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22]
NUM_JOINTS = len(KEEP_21_INDICES)

BONE_PAIRS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),         # Index
    (0, 9), (9, 10), (10, 11), (11, 12),    # Middle
    (0, 13), (13, 14), (14, 15), (15, 16),  # Ring
    (0, 17), (17, 18), (18, 19), (19, 20)   # Pinky
]
PARENT_INDICES = torch.tensor([p for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
CHILD_INDICES = torch.tensor([c for p, c in BONE_PAIRS], dtype=torch.long, device=DEVICE)
DISTAL_BONE_INDICES = [3, 7, 11, 15, 19]
FINGERTIP_JOINTS = [4, 8, 12, 16, 20]
MCP_JOINTS = [1, 5, 9, 13, 17]

# =============================================================================
# 2. SELF-HEALING VRAM INGESTION (LOADS IF NOT ALREADY IN VRAM)
# =============================================================================
def build_file_map(base_dir: Path):
    fmap = {}
    search_dirs = [base_dir]
    if base_dir.parent.exists():
        search_dirs.append(base_dir.parent)
    if Path("/content").exists():
        search_dirs.append(Path("/content"))
    for sdir in search_dirs:
        for p in sdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".json"]:
                if p.name not in fmap:
                    fmap[p.name] = p
    return fmap

def resolve_file(base_dir: Path, rel_path: str, file_map: dict, fallback_subdir: str = "") -> Path:
    rel_clean = str(rel_path).replace("\\", "/").strip()
    p = Path(rel_clean)
    if p.is_absolute() and p.exists():
        return p
    p1 = base_dir / rel_clean
    if p1.exists():
        return p1
    p2 = base_dir.parent / rel_clean
    if p2.exists():
        return p2
    if rel_clean.startswith("dataset/"):
        p3 = base_dir / rel_clean[len("dataset/"):]
        if p3.exists():
            return p3
    fname = Path(rel_clean).name
    if fallback_subdir:
        p4 = base_dir / fallback_subdir / fname
        if p4.exists():
            return p4
        p5 = base_dir.parent / fallback_subdir / fname
        if p5.exists():
            return p5
    if fname in file_map:
        return file_map[fname]
    raise FileNotFoundError(f"Cannot resolve {rel_path} anywhere in {base_dir}")

def extract_frame_id(path_str: str) -> int:
    nums = re.findall(r"\d+", Path(path_str).stem)
    return int(nums[-1])

if 'vram_images' not in globals() or 'vram_targets' not in globals() or len(globals().get('vram_images', [])) == 0:
    print("Building file map across filesystem...")
    FILE_MAP = build_file_map(DATA_DIR)
    print(f"Indexed {len(FILE_MAP):,} candidate files.")
    print("Staging 5,114 frames into Tesla T4 VRAM (compact uint8)...")

    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        records = list(csv.DictReader(f))

    raw_imgs, raw_targs, raw_segs, raw_fids = [], [], [], []
    errors = []
    for r in records:
        try:
            img_p = resolve_file(DATA_DIR, r["image_path"], FILE_MAP, "images")
            kp_p = resolve_file(DATA_DIR, r["orig_kp_right"], FILE_MAP, "keypoints_right")
            bgr = cv2.imread(str(img_p))
            if bgr is None:
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb_resized = cv2.resize(rgb, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)

            with open(kp_p, "r", encoding="utf-8") as kf:
                full_kps = np.array(json.load(kf), dtype=np.float32)

            kps_21 = full_kps[KEEP_21_INDICES]
            kps_rel = (kps_21 - kps_21[0:1]).flatten()
            fid = int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])

            raw_imgs.append(rgb_resized.transpose(2, 0, 1))
            raw_targs.append(kps_rel)
            raw_segs.append(r["segment"])
            raw_fids.append(fid)
        except Exception as e:
            if len(errors) < 3:
                errors.append(str(e))
            continue

    if len(raw_imgs) == 0:
        raise RuntimeError("Failed to load any images! Sample errors:\n" + "\n".join(errors))

    vram_images = torch.tensor(np.stack(raw_imgs), dtype=torch.uint8, device=DEVICE)
    vram_targets = torch.tensor(np.stack(raw_targs), dtype=torch.float32, device=DEVICE)
    sample_segments = np.array(raw_segs)
    sample_fids = np.array(raw_fids)
    unique_segments = np.unique(sample_segments)

    del raw_imgs, raw_targs, raw_segs, raw_fids
    gc.collect()

    vram_mb = (vram_images.untyped_storage().nbytes() + vram_targets.untyped_storage().nbytes()) / (1024 * 1024)
    print(f"Successfully staged {len(vram_images):,} frames into VRAM ({vram_mb:.1f} MB used).")
else:
    print(f"Reusing {len(vram_images):,} frames already resident in VRAM.")
    if 'sample_fids' not in globals() or len(sample_fids) != len(vram_images):
        with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
            manifest_rows = list(csv.DictReader(f))
        sample_fids = np.array([int(r["frame_id"]) if "frame_id" in r else extract_frame_id(r["image_path"])
                                for r in manifest_rows[:len(vram_images)]])

seg_to_indices = defaultdict(list)
for idx, seg in enumerate(sample_segments):
    seg_to_indices[seg].append(idx)

N_SAMPLES = len(vram_images)
prev1_indices = np.zeros(N_SAMPLES, dtype=np.int64)
delta_f_arr = np.zeros(N_SAMPLES, dtype=np.float32)

for seg, indices in seg_to_indices.items():
    indices.sort(key=lambda i: sample_fids[i])
    for step_idx, i in enumerate(indices):
        if step_idx == 0:
            prev1_indices[i] = i
            delta_f_arr[i] = 999.0
        else:
            p_i = indices[step_idx - 1]
            df = float(sample_fids[i] - sample_fids[p_i])
            delta_f_arr[i] = df
            prev1_indices[i] = p_i if df <= 6.5 else i

vram_prev1 = torch.tensor(prev1_indices, dtype=torch.long, device=DEVICE)
vram_contig_flag = torch.tensor((delta_f_arr <= 6.5).astype(np.float32), device=DEVICE)
vram_dt_scale = torch.tensor(np.where(delta_f_arr <= 6.5, delta_f_arr / 3.0, 0.0).astype(np.float32), device=DEVICE)

with torch.no_grad():
    all_kps = vram_targets.view(-1, 21, 3)
    b_all = all_kps[:, CHILD_INDICES] - all_kps[:, PARENT_INDICES]
    L_CANON = torch.norm(b_all, dim=-1).mean(dim=0)

# =============================================================================
# 3. GPU SOBEL & DETERMINISTIC GEOMETRIC RELATIONSHIP TENSOR R_t = R(K_t)
# =============================================================================
class GPUSobelEngine(nn.Module):
    def __init__(self, device: torch.device):
        super().__init__()
        kx = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], device=device).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], device=device).view(1, 1, 3, 3)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    @torch.no_grad()
    def forward(self, rgb_f: torch.Tensor) -> torch.Tensor:
        gray = 0.299 * rgb_f[:, 0:1] + 0.587 * rgb_f[:, 1:2] + 0.114 * rgb_f[:, 2:3]
        gx = F.conv2d(gray, self.kx, padding=1) / 1020.0
        gy = F.conv2d(gray, self.ky, padding=1) / 1020.0
        return torch.sqrt(gx ** 2 + gy ** 2 + 1e-12)

gpu_extractor = GPUSobelEngine(DEVICE)

def rotation_6d_to_matrix(d6: torch.Tensor) -> torch.Tensor:
    a1, a2 = d6[:, :3], d6[:, 3:]
    b1 = F.normalize(a1, dim=-1, eps=1e-7)
    dot = (b1 * a2).sum(dim=-1, keepdim=True)
    b2 = F.normalize(a2 - dot * b1, dim=-1, eps=1e-7)
    b3 = torch.cross(b1, b2, dim=-1)
    return torch.stack([b1, b2, b3], dim=-1)

def reconstruct_joints_from_bones(bone_vectors: torch.Tensor) -> torch.Tensor:
    B = bone_vectors.shape[0]
    joints = torch.zeros((B, 21, 3), dtype=bone_vectors.dtype, device=bone_vectors.device)
    for idx, (p, c) in enumerate(BONE_PAIRS):
        joints[:, c] = joints[:, p] + bone_vectors[:, idx]
    return joints

def compute_relationship_tensor_R(joints: torch.Tensor, bone_dirs: torch.Tensor) -> torch.Tensor:
    """
    Deterministic SE(3)-invariant & camera-aware Geometric Relationship Tensor R_t (03_observation/relationship.py):
      1. Pairwise 5x5 fingertip distances (10 unique pairs, normalized by 100mm) -> 10D
      2. Fingertip-to-MCP distances (5 fingers x 5 MCPs = 25D, normalized by 100mm) -> 25D
      3. Intra-finger sequential bone dot products (cos_theta for 5 fingers x 3 joints = 15D) -> 15D
      4. Adjacent-finger parallel phalanx dot products (4 adjacent pairs x 4 bone levels = 16D) -> 16D
      5. Signed palmar elevation of all 5 fingertips relative to palm normal (5D) -> 5D
      Total Relationship State R_t = 71D
    """
    tips = joints[:, FINGERTIP_JOINTS, :]  # (B, 5, 3)
    mcps = joints[:, MCP_JOINTS, :]        # (B, 5, 3)

    # 1. Pairwise fingertip distances (10D)
    tip_diff = tips.unsqueeze(2) - tips.unsqueeze(1)  # (B, 5, 5, 3)
    tip_dists = torch.norm(tip_diff + 1e-7, dim=-1) / 100.0
    tri_u = torch.triu_indices(5, 5, offset=1, device=joints.device)
    r_tip_pairs = tip_dists[:, tri_u[0], tri_u[1]]    # (B, 10)

    # 2. Fingertip to MCP distances (25D)
    tip_mcp_diff = tips.unsqueeze(2) - mcps.unsqueeze(1)
    r_tip_mcp = (torch.norm(tip_mcp_diff + 1e-7, dim=-1) / 100.0).reshape(-1, 25)

    # 3. Intra-finger sequential dot products (15D)
    dirs_5x4 = bone_dirs.reshape(-1, 5, 4, 3)
    r_intra_cos = (dirs_5x4[:, :, 0:3, :] * dirs_5x4[:, :, 1:4, :]).sum(dim=-1).reshape(-1, 15)

    # 4. Cross-finger adjacent bone dot products (16D)
    r_inter_cos = (dirs_5x4[:, 0:4, :, :] * dirs_5x4[:, 1:5, :, :]).sum(dim=-1).reshape(-1, 16)

    # 5. Signed palmar elevation of fingertips relative to palm normal (5D)
    u_idx = bone_dirs[:, 4]
    u_mid = bone_dirs[:, 8]
    u_pky = bone_dirs[:, 16]
    palm_norm = F.normalize(torch.cross(u_mid, u_pky - u_idx, dim=-1), dim=-1, eps=1e-7)  # (B, 3)
    r_palm_elev = ((tips - mcps[:, 2:3, :]) * palm_norm.unsqueeze(1)).sum(dim=-1) / 100.0  # (B, 5)

    return torch.cat([r_tip_pairs, r_tip_mcp, r_intra_cos, r_inter_cos, r_palm_elev], dim=-1)  # (B, 71)

# =============================================================================
# 4. ARCHITECTURES: BASELINE LCF vs. CLOSED-LOOP (K + R) & TEMPORAL STATE REFINER
# =============================================================================
class BaseLCFNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((6, 8))
        )
        self.shared_fc = nn.Sequential(
            nn.Linear(256 * 6 * 8, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1)
        )
        self.meta_head = nn.Linear(512, 5 * 3)
        self.latent_frame_head = nn.Linear(512, 6)
        self.phalanx_head = nn.Linear(512, 15 * 3)
        self.register_buffer("bone_lens", L_CANON)

    def decode_lcf(self, feat: torch.Tensor):
        B = feat.shape[0]
        meta_cam_u = F.normalize(self.meta_head(feat).reshape(B, 5, 3), dim=-1, eps=1e-7)
        R_latent = rotation_6d_to_matrix(self.latent_frame_head(feat))
        phalanx_local_u = F.normalize(self.phalanx_head(feat).reshape(B, 15, 3), dim=-1, eps=1e-7)
        phalanx_cam_u = torch.bmm(R_latent, phalanx_local_u.transpose(1, 2)).transpose(1, 2)

        bone_dirs = torch.zeros((B, 20, 3), dtype=feat.dtype, device=feat.device)
        bone_dirs[:, 0] = meta_cam_u[:, 0]
        bone_dirs[:, 1:4] = phalanx_cam_u[:, 0:3]
        finger_offsets = [4, 8, 12, 16]
        for f_idx in range(4):
            b_off = finger_offsets[f_idx]
            p_off = 3 + f_idx * 3
            bone_dirs[:, b_off] = meta_cam_u[:, f_idx + 1]
            bone_dirs[:, b_off + 1:b_off + 4] = phalanx_cam_u[:, p_off:p_off + 3]

        bone_vecs = bone_dirs * self.bone_lens.view(1, 20, 1)
        joints = reconstruct_joints_from_bones(bone_vecs)
        return joints, bone_dirs, R_latent, phalanx_local_u

    def forward(self, g_t: torch.Tensor):
        E_t = self.shared_fc(torch.flatten(self.encoder(g_t), 1))
        joints, bone_dirs, _, _ = self.decode_lcf(E_t)
        return joints, bone_dirs, E_t


class ClosedLoopKRRefiner(nn.Module):
    """
    Implements the hm_ex Definitive Specification (source: 48):
      - Takes Visual Evidence E_t (512D) + Initial Skeleton K_t^0 (63D) + Relationship R_t^0 (71D)
      - Optionally includes contiguous prior state context (K_{t-1}, R_{t-1}, Dual-Edge E_t - E_{t-1})
      - Outputs articulated latent residual updates through the exact L_CANON bottleneck (0.00mm bone stretch).
    """
    def __init__(self, use_temporal_state: bool = False):
        super().__init__()
        self.use_temporal_state = use_temporal_state
        self.base_lcf = BaseLCFNetwork()

        # Input tuple: E_t (512) + K_t (63) + B_t (60) + R_t (71) = 706D
        in_dim = 512 + 63 + 60 + 71
        if use_temporal_state:
            # Add ΔE_t (512) + ΔK_t (63) + ΔR_t (71) + dt_flag (2) = +648D
            in_dim += 512 + 63 + 71 + 2

        self.residual_mlp = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 512)
        )
        # Zero-init final layer so epoch 0 starts cleanly at the proven LCF baseline
        nn.init.zeros_(self.residual_mlp[-1].weight)
        nn.init.zeros_(self.residual_mlp[-1].bias)

    def forward(self, g_t: torch.Tensor, g_prev: torch.Tensor = None,
                contig_flag: torch.Tensor = None, dt_scale: torch.Tensor = None):
        j0, d0, E_t = self.base_lcf(g_t)
        K_0 = j0.reshape(-1, 63) / 100.0
        B_0 = d0.reshape(-1, 60)
        R_0 = compute_relationship_tensor_R(j0, d0)

        if not self.use_temporal_state:
            state_tuple = torch.cat([E_t, K_0, B_0, R_0], dim=-1)
        else:
            with torch.no_grad():
                j_prev, d_prev, E_prev = self.base_lcf(g_prev)
                K_prev = j_prev.reshape(-1, 63) / 100.0
                R_prev = compute_relationship_tensor_R(j_prev, d_prev)

            c_mask = contig_flag.unsqueeze(-1)  # (B, 1): 1 if df <= 6 else 0
            dt_col = dt_scale.unsqueeze(-1)     # (B, 1)

            dE = (E_t - E_prev) * c_mask
            dK = (K_0 - K_prev) * c_mask
            dR = (R_0 - R_prev) * c_mask
            state_tuple = torch.cat([E_t, K_0, B_0, R_0, dE, dK, dR, c_mask, dt_col], dim=-1)

        delta_E = self.residual_mlp(state_tuple)
        j_ref, d_ref, _, _ = self.base_lcf.decode_lcf(E_t + delta_E)
        return j0, j_ref, d_ref

# =============================================================================
# 5. LOSS EVALUATOR & 3-SPLIT BENCHMARK HARNESS
# =============================================================================
class MetricLossEvaluator:
    def __call__(self, pred: torch.Tensor, target: torch.Tensor):
        l_pos = torch.norm(pred - target, dim=-1).mean()
        b_pred = pred[:, CHILD_INDICES] - pred[:, PARENT_INDICES]
        b_gt = target[:, CHILD_INDICES] - target[:, PARENT_INDICES]
        len_pred = torch.norm(b_pred, dim=-1)
        len_gt = torch.norm(b_gt, dim=-1)
        l_len = F.l1_loss(len_pred, len_gt)
        eps = 1e-7
        u_p = b_pred / (len_pred.unsqueeze(-1) + eps)
        u_g = b_gt / (len_gt.unsqueeze(-1) + eps)
        cos_sim = torch.clamp((u_p * u_g).sum(dim=-1), -1.0, 1.0)
        l_dir = (1.0 - cos_sim).mean()
        return l_pos + 0.5 * l_len + 20.0 * l_dir

eval_engine = MetricLossEvaluator()

SEEDS = [42, 101, 777]
EPOCHS = 15
BATCH_SIZE = 64

CONFIGS = [
    ("1. Single-Pass LCF Baseline (E_t -> LCF)",                  "base"),
    ("2. Closed-Loop K+R Refiner (E_t + K_t + R_t -> ΔLCF)",      "kr_static"),
    ("3. State + Dual-Edge K+R Tracker (E_t + ΔE + K + R + ΔK)",  "kr_temporal"),
]

res_mpjpe = {c[0]: [] for c in CONFIGS}
res_df3 = {c[0]: [] for c in CONFIGS}
res_tips = {c[0]: [] for c in CONFIGS}
res_distal = {c[0]: [] for c in CONFIGS}

print("\n" + "=" * 112)
print(" EXPERIMENT G: BRIDGING hm_ex (K + R RELATIONSHIP & STATE RESIDUAL) TO REAL RE:INTERHAND")
print("=" * 112)

for exp_name, mode in CONFIGS:
    print(f"\n>>> Running {exp_name} <<<")
    for seed in SEEDS:
        torch.manual_seed(seed)
        np.random.seed(seed)

        shuffled_segs = np.random.permutation(unique_segments)
        split_idx = int(0.8 * len(shuffled_segs))
        train_segs = set(shuffled_segs[:split_idx])
        val_segs = set(shuffled_segs[split_idx:])

        train_indices = torch.tensor(np.where(np.isin(sample_segments, list(train_segs)))[0], device=DEVICE)
        val_indices = torch.tensor(np.where(np.isin(sample_segments, list(val_segs)))[0], device=DEVICE)

        if mode == "base":
            model = BaseLCFNetwork().to(DEVICE)
        elif mode == "kr_static":
            model = ClosedLoopKRRefiner(use_temporal_state=False).to(DEVICE)
        else:
            model = ClosedLoopKRRefiner(use_temporal_state=True).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        for epoch in range(1, EPOCHS + 1):
            model.train()
            perm = train_indices[torch.randperm(len(train_indices))]
            for i in range(0, len(perm), BATCH_SIZE):
                b_idx = perm[i:i + BATCH_SIZE]
                g_t = gpu_extractor(vram_images[b_idx].float())
                y_batch = vram_targets[b_idx].view(-1, 21, 3)

                optimizer.zero_grad()
                if mode == "base":
                    pred_j, _, _ = model(g_t)
                    loss = eval_engine(pred_j, y_batch)
                elif mode == "kr_static":
                    j0, j_ref, _ = model(g_t)
                    # Deep supervision on both coarse Stage-0 and refined Stage-1
                    loss = 0.4 * eval_engine(j0, y_batch) + 0.6 * eval_engine(j_ref, y_batch)
                else:
                    p_idx = vram_prev1[b_idx]
                    g_prev = gpu_extractor(vram_images[p_idx].float())
                    c_flag = vram_contig_flag[b_idx]
                    dt_s = vram_dt_scale[b_idx]
                    j0, j_ref, _ = model(g_t, g_prev, c_flag, dt_s)
                    loss = 0.4 * eval_engine(j0, y_batch) + 0.6 * eval_engine(j_ref, y_batch)

                loss.backward()
                optimizer.step()

        model.eval()
        val_preds, val_gts, val_dfs = [], [], []
        with torch.no_grad():
            for i in range(0, len(val_indices), BATCH_SIZE):
                b_idx = val_indices[i:i + BATCH_SIZE]
                g_t = gpu_extractor(vram_images[b_idx].float())
                if mode == "base":
                    pj, _, _ = model(g_t)
                elif mode == "kr_static":
                    _, pj, _ = model(g_t)
                else:
                    p_idx = vram_prev1[b_idx]
                    g_prev = gpu_extractor(vram_images[p_idx].float())
                    c_flag = vram_contig_flag[b_idx]
                    dt_s = vram_dt_scale[b_idx]
                    _, pj, _ = model(g_t, g_prev, c_flag, dt_s)

                val_preds.append(pj.cpu().numpy())
                val_gts.append(vram_targets[b_idx].view(-1, 21, 3).cpu().numpy())
                val_dfs.append(delta_f_arr[b_idx.cpu().numpy()])

        all_p = np.concatenate(val_preds, axis=0)
        all_g = np.concatenate(val_gts, axis=0)
        all_df = np.concatenate(val_dfs, axis=0)

        mpjpe = np.linalg.norm(all_p - all_g, axis=-1).mean()
        df3_mask = all_df <= 3.5
        mpjpe_df3 = np.linalg.norm(all_p[df3_mask] - all_g[df3_mask], axis=-1).mean()
        tip_err = np.linalg.norm(all_p[:, FINGERTIP_JOINTS] - all_g[:, FINGERTIP_JOINTS], axis=-1).mean()

        b_p = all_p[:, CHILD_INDICES.cpu().numpy()] - all_p[:, PARENT_INDICES.cpu().numpy()]
        b_g = all_g[:, CHILD_INDICES.cpu().numpy()] - all_g[:, PARENT_INDICES.cpu().numpy()]
        u_p = b_p / (np.linalg.norm(b_p, axis=-1, keepdims=True) + 1e-7)
        u_g = b_g / (np.linalg.norm(b_g, axis=-1, keepdims=True) + 1e-7)
        cos_sim = np.clip(np.sum(u_p * u_g, axis=-1), -1.0, 1.0)
        deg_bones = np.arccos(cos_sim) * (180.0 / np.pi)
        distal_deg = deg_bones[:, DISTAL_BONE_INDICES].mean()

        res_mpjpe[exp_name].append(mpjpe)
        res_df3[exp_name].append(mpjpe_df3)
        res_tips[exp_name].append(tip_err)
        res_distal[exp_name].append(distal_deg)

        print(f"  [Seed {seed:<3}] All MPJPE: {mpjpe:5.2f} mm | Adj(df=3): {mpjpe_df3:5.2f} mm | "
              f"Fingertips: {tip_err:5.2f} mm | Distal Dir: {distal_deg:4.1f}°")

        del model, optimizer
        torch.cuda.empty_cache()

print("\n" + "=" * 116)
print(" EXPERIMENT G SUMMARY: CLOSED-LOOP (K + R) RELATIONSHIP & STATE REFINEMENT ACROSS 3 SPLITS")
print("=" * 116)
print(f"{'Model Configuration':<58} | {'All MPJPE':<15} | {'Adj(df=3) MPJPE':<16} | {'Fingertips':<15} | {'Distal Dir'}")
print("-" * 116)
for exp_name, _ in CONFIGS:
    mp, sp = np.mean(res_mpjpe[exp_name]), np.std(res_mpjpe[exp_name])
    m3, s3 = np.mean(res_df3[exp_name]), np.std(res_df3[exp_name])
    mt, st = np.mean(res_tips[exp_name]), np.std(res_tips[exp_name])
    md, sd = np.mean(res_distal[exp_name]), np.std(res_distal[exp_name])
    print(f"{exp_name:<58} | {mp:5.2f}±{sp:4.2f} mm  | {m3:5.2f}±{s3:4.2f} mm   | {mt:5.2f}±{st:4.2f} mm  | {md:4.1f}°±{sd:3.1f}°")
print("=" * 116)